# RQ ii — Cross-Platform Sentiment and Emotion Analysis
<!-- structured-notebook -->
## Notebook Summary
Purpose: answer the second research question — how does sentiment vary between platforms? — by computing per-document sentiment and emotion distributions uniformly across PubMed, news, Reddit, YouTube, and Telegram, then comparing by topic, by year, and across the COVID-19 intervention point.

Main steps:
- Map topics to human-readable names across all platforms so cross-platform comparisons are interpretable.
- Compute per-document sentiment (RoBERTa 3-class) and emotion (GoEmotions 27-class) probabilities.
- Aggregate by topic × platform × year using length-weighted averaging.
- Produce sentiment-over-time line plots, topic × year heatmaps, per-platform emotion distributions.
- Compute "emotional certainty" as the mean probability of each document's top emotion.
- Run COVID pre/post comparison with grouped bars, line plots, and pie charts over two window schemes (2-2-2 and 10-2-3).


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/topic_matchings/<pair>/stable_pairs_readable.csv` | Produced by `04_topic_matching/02_finalizing_pairs.ipynb` |
| Input | `<DATA_ROOT>/topic_matchings/<platform>_with_topics.csv` | Produced by `04_topic_matching/05_gathering_keywords.ipynb` |
| Input | `<DATA_ROOT>/Reddit/output/comments/discussion_post_comment_sentiment.csv` | Produced by `05_comment_analysis/01_comment_attaching.ipynb` |
| Output | Per-platform / per-topic / per-year sentiment + emotion CSVs and figures | Published in paper §4 / §5 |


# Model Choices
<!-- structured-notebook -->
## Summary

### Sentiment: `cardiffnlp/twitter-roberta-base-sentiment-latest`

3-class output: positive, negative, neutral. Selected because:

1. Fine-tuned on social media text (TweetEval benchmark) — generalizes well from Reddit to PubMed abstracts when the abstract reads conversationally.
2. RoBERTa-base scale fits on a single GPU and runs on CPU in a reasonable time for the full corpus.
3. Consistency across platforms: using the same model everywhere removes a confound when comparing platforms.
4. Length-weighted aggregation: per-document scores are averaged by topic × platform × year weighted by document length so long-form PubMed abstracts do not get drowned out by short Reddit posts.

### Emotion: `joeddav/distilbert-base-uncased-go-emotions-student`

27 emotion categories (positive: admiration, amusement, approval, caring, desire, gratitude, joy, love, optimism, pride, relief; neutral: curiosity, realization, surprise, neutral, confusion; negative: anger, annoyance, disappointment, disapproval, disgust, embarrassment, fear, grief, nervousness, remorse, sadness).

Selected because:

1. Distilled student model — fast enough to run on the full combined corpus.
2. 27-class fine-grained output gives more narrative detail than 6 Ekman-basic emotions.
3. Probability distribution per document, not a single label — enables the "emotional certainty" metric (mean top-emotion probability).
4. Same length-weighted aggregation scheme as sentiment.


# Output Artifacts
<!-- structured-notebook -->
## Summary

| File | Contents |
|---|---|
| `sentiment_per_doc.csv` | Per-document sentiment scores (`positive`, `negative`, `neutral`) |
| `sentiment_by_platform.csv` | Aggregated by platform |
| `sentiment_by_topic_platform.csv` | Aggregated by topic × platform |
| `topic_sentiment_timeseries_years_emotions/` | Per-topic temporal emotion series |
| `topic_sentiment_timeseries_years_roberta/` | RoBERTa sentiment time series |
| `topic_sentiment_timeseries_years_vader/` | VADER sentiment time series (lightweight comparison baseline) |
| `fig_sentiment_heatmaps/` | Topic × year sentiment heatmap figures |

The VADER outputs are a sanity-check alternative — a lightweight lexicon-based sentiment model that does not require a GPU. Agreement between VADER and RoBERTa gives confidence that the RoBERTa outputs are not dominated by model-specific quirks.


# COVID Pre/Post Framing
<!-- structured-notebook -->
## Summary

The notebook contains three COVID-period analyses:

1. **Grouped bars** — per-emotion average probability in pre vs. post windows.
2. **Line plots** — time series of sentiment / emotion through the intervention point.
3. **Pie charts** — proportional breakdowns pre vs. post.

Two window schemes are used:

| Scheme | Pre | Intervention | Post |
|---|---|---|---|
| 2-2-2 | 2018–2019 | 2020–2021 | 2022–2023 |
| 10-2-3 | 2010–2019 | 2020–2021 | 2022–2024 |

The 2-2-2 scheme isolates near-term effects; the 10-2-3 scheme places the intervention against a longer historical baseline. Formal Interrupted Time Series (ITS) with multi-criteria sensitivity (3 FDR × 3 Cohen's d thresholds) is published — see the paper §6.3. That formal ITS is not reproduced here; this notebook carries the descriptive comparison that grounds the ITS.


# Bio-Optimistic Mediatization Loop (Narrative Framing)
<!-- structured-notebook -->
## Summary

The interpretive frame that ties together the RQ ii findings in the paper's discussion section:

> Taken together, these dynamics constitute what may be described as a **bio-optimistic mediatization loop** — a feedback system in which scientific discovery, lifestyle media, and participatory communities continually reinforce one another through optimistic, emotionally moderate, advice-oriented framings of human enhancement. Within this loop, longevity discourse functions simultaneously as biomedical knowledge, media narrative, and cultural aspiration. The result is a self-sustaining ecosystem of constructive affect that bridges expert and lay communication, transforming healthy ageing from a specialized research topic into a shared cultural horizon.

This framing connects the quantitative finding (sentiment is broadly positive-to-neutral with higher emotional certainty on participatory platforms) to the qualitative argument about mediatization. Future work would quantify how credibility signals, emotional framing, and topic diffusion jointly influence health behavior and public trust in science across media ecosystems.


# Completion State
<!-- structured-notebook -->
## Summary

| Feature | Status | Notes |
|---|---|---|
| RoBERTa 3-class sentiment per document | Done | Length-weighted aggregation by topic × platform × year |
| GoEmotions 27-class emotion per document | Done | Probability distribution per document |
| Sentiment over time per platform | Done | Yearly resolution — monthly was too sparse for PubMed |
| Topic × year sentiment heatmaps | Done | Per platform |
| Emotion distribution over time | Done | Per-platform and per-document views |
| Emotional certainty (mean top-emotion probability) | Done | Used as confidence metric in paper discussion |
| Sectional emotion histogram (years × probability) | Done | From full `emotions.csv` |
| COVID-period comparison (descriptive) | Done | Grouped bars, line plots, pie charts; 2-2-2 and 10-2-3 windows |
| "Bio-optimistic mediatization loop" framing | Done (narrative) | Interprets findings for paper discussion |

### Follow-up items (not blocking the paper)

1. Effect-size CIs on COVID pre/post comparisons — current plots have no explicit Cliff's delta / Cohen's d with 95% CIs as a single summary figure.
2. Wordshift / rank-divergence overlays per cluster — would connect RQ ii to RQ v without doing the full vocabulary-shift analysis.
3. Per-platform emotion entropy as a complementary certainty metric.
4. Emotion-trajectory clustering — group topics by their temporal emotion signature rather than visualizing per-topic series.


In [ ]:
from src.shared_reddit_telegram.config import (
    TOPIC_MATCHINGS,
)
from typing import Union

pip install vaderSentiment

In [ ]:
# Re-run with a fix for mixed-type topic indices (cast to string for grouping/plotting)
import os
import re
import math
import unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#from caas_jupyter_tools import display_dataframe_to_user

def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\n", " ").replace("\r", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s

def merge_text_columns(df: pd.DataFrame, columns: list, separator: str = " ") -> pd.Series:
    """Merge multiple text columns into one, filtering out NaN/empty values."""
    result = []
    for idx, row in df.iterrows():
        parts = []
        for col in columns:
            if col in df.columns:
                val = row[col]
                if pd.notna(val) and str(val).strip():
                    parts.append(str(val).strip())
        result.append(separator.join(parts))
    return pd.Series(result, index=df.index)

def build_topic_mapping(reddit_path: str) -> dict:
    """Build a mapping from topic_num to topic name using Reddit CSV as reference."""
    try:
        df = pd.read_csv(reddit_path)
        if 'topic_num' in df.columns and 'topic' in df.columns:
            # Get unique topic_num to topic mappings
            mapping_df = df[['topic_num', 'topic']].drop_duplicates()
            topic_map = dict(zip(mapping_df['topic_num'], mapping_df['topic']))
            print(f"Built topic mapping with {len(topic_map)} topics from Reddit")
            return topic_map
    except Exception as e:
        print(f"Warning: Could not build topic mapping from Reddit: {e}")
    return {}

def choose_text_column(df: pd.DataFrame, platform: str) -> str:
    """Choose which column(s) to use for text. Returns column name or list of columns."""
    candidates = []
    if platform == "Reddit":
        candidates = ["title", "selftext"]
    if platform == "YouTube":
        candidates = ["title", "description", "comments", "keywords"]
    if platform == "PubMed":
        return "title_abstract"
    if platform == "News":
        # Return multiple columns to merge
        candidates = ["Title", "Abstract"]
    available = [c for c in candidates if c in df.columns]
    if len(available) > 1:
        return available  # Return list to signal merging
    elif available:
        return available[0]
    candidates = [
        "title_abstract", "text", "content", "body", "abstract", "description",
        "selftext", "clean_text", "full_text", "title_and_abstract", "title",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    obj_cols = [c for c in df.columns if df[c].dtype == "O"]
    if obj_cols:
        return obj_cols[0]
    return df.columns[0]

class OfflineLexiconSentiment:
    def __init__(self):
        pos = """
        good great excellent positive promising breakthrough success improvement effective
        beneficial robust safe hopeful optimistic innovative advance longevity youthful
        efficient helpful heal healthy resilient progress win supportive enhance
        """
        neg = """
        bad poor worse worst negative risk risky harm harmful danger dangerous failure
        unsafe concern concerning fear fearful alarming scandal crisis crisis-level
        uncertain controversy controversial doubt skeptical side-effect adverse aging
        decline degrade toxic
        """
        self.pos = set(w.strip().lower() for w in pos.split())
        self.neg = set(w.strip().lower() for w in neg.split())
        self.negators = {"no", "not", "never", "without", "hardly", "barely", "rarely"}
        self.token_re = re.compile(r"[A-Za-z][A-Za-z'\-]+")

    def polarity(self, text: str) -> float:
        text = normalize_text(text).lower()
        tokens = self.token_re.findall(text)
        if not tokens:
            return 0.0
        score = 0.0
        window = 3
        for i, tok in enumerate(tokens):
            val = 0.0
            if tok in self.pos:
                val = 1.0
            elif tok in self.neg:
                val = -1.0
            if val != 0.0:
                start = max(0, i - window)
                if any(t in self.negators for t in tokens[start:i]):
                    val = -val
                score += val
        score = score / max(1.0, math.sqrt(len(tokens)))
        return float(max(-1.0, min(1.0, score / 3.0)))

def get_sentiment_fn():
    # Try VADER via NLTK first
    try:
        import nltk
        from nltk.sentiment import SentimentIntensityAnalyzer

        try:
            analyzer = SentimentIntensityAnalyzer()
        except LookupError as e:
            # The vader_lexicon is missing; try to fetch (comment these 2 lines out if offline)
            try:
                nltk.download('vader_lexicon', quiet=True)
                analyzer = SentimentIntensityAnalyzer()
            except Exception as dl_e:
                print(f"[warn] VADER lexicon unavailable and download failed: {dl_e}")
                raise

        def vader_score(text: str) -> float:
            text = normalize_text(text)
            try:
                vs = analyzer.polarity_scores(text)
                return float(vs.get("compound", 0.0))

            except Exception as inner_e:
                # If per-text scoring fails, treat as neutral instead of crashing
                return 0.0

        # Smoke-test
        _ = vader_score("This is great!")
        print("[info] Using VADER via NLTK")
        return vader_score, "vader"

    except Exception as e:
        print(f"[warn] NLTK VADER unavailable: {e}")

    # Try vaderSentiment (no NLTK resource)
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        analyzer = SentimentIntensityAnalyzer()

        def vader_sentiment_score(text: str) -> float:
            text = normalize_text(text)
            try:
                vs = analyzer.polarity_scores(text)
                return float(vs.get("compound", 0.0))
            except Exception:
                return 0.0

        _ = vader_sentiment_score("This is great!")
        print("[info] Using vaderSentiment package")
        return vader_sentiment_score, "vaderSentiment"

    except Exception as e:
        print(f"[warn] vaderSentiment unavailable: {e}")

    # Fallback: offline lexicon (always works)
    offline = OfflineLexiconSentiment()
    print("[info] Using offline lexicon fallback")
    return offline.polarity, "offline_lexicon"

sentiment_fn, sentiment_backend = get_sentiment_fn()
print(sentiment_backend)

def load_with_platform(path, platform_name):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    text_col = choose_text_column(df, platform_name)

    # Handle merging multiple columns if needed
    if isinstance(text_col, list):
        df["_text"] = merge_text_columns(df, text_col, separator=" ")
    else:
        df["_text"] = df[text_col].astype(str).map(normalize_text)

    topic_col = None
    for c in ["topic_num", "topic", "topic_id", "dominant_topic", "Topic", "topic_label"]:
        if c in df.columns:
            topic_col = c
            break
    if topic_col is None:
        df["topic"] = -1
        topic_col = "topic"
    df["_topic"] = df[topic_col]
    df["_platform"] = platform_name
    keep = ["_platform", "_topic", "_text"]
    for c in ["date", "created_utc", "published_at", "pub_date", "timestamp"]:
        if c in df.columns:
            df["_date"] = df[c]
            keep.append("_date")
            break
    return df[keep].rename(columns={"_platform":"platform","_topic":"topic","_text":"text"})

# Build topic mapping from Reddit first
reddit_path = TOPIC_MATCHINGS / "reddit_with_topics.csv"
topic_num_to_name = build_topic_mapping(reddit_path)

datasets = []
paths = [
    (TOPIC_MATCHINGS / "pubmed_with_topics.csv", "PubMed"),
    (TOPIC_MATCHINGS / "news_with_topics.csv", "News"),
    (TOPIC_MATCHINGS / "reddit_with_topics.csv", "Reddit"),
    (TOPIC_MATCHINGS / "youtube_with_topics.csv", "YouTube"),
]
loaded_ok = []
for pth, name in paths:
    if os.path.exists(pth):
        try:
            datasets.append(load_with_platform(pth, name))
            loaded_ok.append(name)
        except Exception as e:
            print(f"Failed to load {name} from {pth}: {e}")
    else:
        print(f"Missing: {pth}")

if not datasets:
    raise RuntimeError("No datasets were loaded.")

df_all = pd.concat(datasets, ignore_index=True)

# Map topic numbers to topic names if mapping is available
if topic_num_to_name:
    df_all["topic_name"] = df_all["topic"].map(topic_num_to_name)
    # For topics without a name mapping, use the original topic value
    df_all["topic_name"] = df_all["topic_name"].fillna(df_all["topic"].astype(str))
    print(f"Mapped {df_all['topic_name'].notna().sum()} topics to names")
else:
    df_all["topic_name"] = df_all["topic"].astype(str)

def safe_sentiment(s):
    try:
        return sentiment_fn(s)
    except Exception:
        return 0.0

df_all["sentiment"] = df_all["text"].astype(str).map(safe_sentiment)
def label_from_score(x: float) -> str:
    if x >= 0.05:
        return "positive"
    elif x <= -0.05:
        return "negative"
    return "neutral"
df_all["sentiment_label"] = df_all["sentiment"].map(label_from_score)

# Ensure topic is string for grouping and plotting - use topic_name
df_all["topic_str"] = df_all["topic_name"].astype(str)

# Save per-doc
out_per_doc = "sentiment_per_doc.csv"
df_all.to_csv(out_per_doc, index=False)

# Aggregations
by_platform = (
    df_all.groupby("platform")["sentiment"]
    .agg(["count", "mean", "std", "median"])
    .reset_index()
    .sort_values("mean", ascending=False)
)
label_dist = (
    df_all.pivot_table(index="platform", columns="sentiment_label", values="text", aggfunc="count", fill_value=0)
    .reset_index()
)
by_topic_platform = (
    df_all.groupby(["platform", "topic_str"])["sentiment"]
    .agg(["count", "mean", "std", "median"])
    .reset_index()
    .sort_values(["platform", "mean"], ascending=[True, False])
)

out_by_platform = "sentiment_by_platform.csv"
out_by_topic_platform = "sentiment_by_topic_platform.csv"
by_platform.to_csv(out_by_platform, index=False)
by_topic_platform.to_csv(out_by_topic_platform, index=False)

# Visualizations
# 1) Boxplot by platform
plt.figure(figsize=(8, 5))
order = by_platform["platform"].tolist()
data = [df_all.loc[df_all["platform"] == p, "sentiment"].values for p in order]
plt.boxplot(data, labels=order, showmeans=True)
plt.title("Sentiment Distribution by Platform")
plt.xlabel("Platform")
plt.ylabel("Sentiment (score in [-1, 1])")
plt.tight_layout()
plot1 = "plot_sentiment_by_platform.png"
plt.savefig(plot1, dpi=150)
plt.close()

# 2) Heatmap mean sentiment topic x platform (top N topics)
N_TOPICS = 30
topic_counts = (
    df_all.groupby("topic_str")["text"].count().sort_values(ascending=False).head(N_TOPICS)
)
top_topics = topic_counts.index.tolist()
heat_df = (
    df_all[df_all["topic_str"].isin(top_topics)]
    .groupby(["topic_str", "platform"])["sentiment"].mean()
    .unstack("platform")
    .fillna(0.0)
    .sort_index()
)

plt.figure(figsize=(14, 10))
plt.imshow(heat_df.values, aspect="auto", interpolation="nearest")
plt.title("Mean Sentiment by Topic × Platform (Top Topics)")
plt.xlabel("Platform")
plt.ylabel("Topic")
plt.xticks(ticks=np.arange(len(heat_df.columns)), labels=list(heat_df.columns), rotation=45, ha="right")
plt.yticks(ticks=np.arange(len(heat_df.index)), labels=[str(t) for t in heat_df.index])
cb = plt.colorbar()
cb.set_label("Mean sentiment")
plt.tight_layout()
plot2 = "plot_topic_platform_heatmap.png"
plt.savefig(plot2, dpi=150)
plt.close()

print("Loaded datasets:", loaded_ok)
print("Sentiment backend used:", sentiment_backend)
print("Saved files:")
print("-", out_per_doc)
print("-", out_by_platform)
print("-", out_by_topic_platform)
print("-", plot1)
print("-", plot2)

Topics over time (normalized per topic-platform)

In [ ]:
# Fix the groupby/apply index handling to avoid duplicate columns on reset_index.

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------- CONFIG ---------
INPUTS = [
    (TOPIC_MATCHINGS / "pubmed_with_topics.csv", "PubMed"),
    (TOPIC_MATCHINGS / "news_with_topics.csv", "News"),
    (TOPIC_MATCHINGS / "reddit_with_topics.csv", "Reddit"),
    (TOPIC_MATCHINGS / "youtube_with_topics.csv", "YouTube"),
]
OUT_DIR = "dont_use/sentiment_over-time"
DATE_START = "2010-01-01"
DATE_END   = "2025-12-31"
FREQ = "MS"  # monthly start

os.makedirs(OUT_DIR, exist_ok=True)

def choose_date_col(df: pd.DataFrame):
    for c in ["date", "Date", "created_utc", "published_at", "pub_date", "timestamp"]:
        if c in df.columns:
            return c
    if "year" in df.columns and "month" in df.columns:
        return ("year","month")
    return None

def parse_date_series(df: pd.DataFrame, date_sel):
    if isinstance(date_sel, tuple):
        y, m = date_sel
        years = pd.to_numeric(df[y], errors="coerce").astype("Int64")
        months = pd.to_numeric(df[m], errors="coerce").astype("Int64")
        return pd.to_datetime(dict(year=years, month=months, day=1), errors="coerce")
    s = df[date_sel]
    if np.issubdtype(s.dtype, np.number):
        try:
            return pd.to_datetime(s, unit="s", errors="coerce")
        except Exception:
            try:
                return pd.to_datetime(s, unit="ms", errors="coerce")
            except Exception:
                return pd.to_datetime(s, errors="coerce")
    else:
        return pd.to_datetime(s, errors="coerce")

def load_platform_df(path, platform):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    topic_col = None
    for c in ["topic", "keywords", "topic_num", "topic_id", "dominant_topic", "Topic", "topic_label"]:
        if c in df.columns:
            topic_col = c
            break
    if topic_col is None:
        df["topic"] = -1
        topic_col = "topic"
    date_sel = choose_date_col(df)
    if date_sel is None:
        df["_date"] = pd.NaT
    else:
        df["_date"] = parse_date_series(df, date_sel)
    out = df[[topic_col, "_date"]].copy()
    out.columns = ["topic", "date"]
    out["platform"] = platform
    out["topic"] = out["topic"].astype(str)
    return out

frames = []
for p, plat in INPUTS:
    if os.path.exists(p):
        frames.append(load_platform_df(p, plat))
    else:
        print(f"Missing: {p}")

if not frames:
    raise RuntimeError("No input CSVs found.")

df = pd.concat(frames, ignore_index=True)
df = df[pd.notna(df["date"])].copy()
df = df[(df["date"] >= DATE_START) & (df["date"] <= DATE_END)].copy()

months = pd.date_range(start=DATE_START, end=DATE_END, freq=FREQ)
df["month"] = df["date"].values.astype("datetime64[M]")

grp = df.groupby(["topic", "platform", "month"]).size().rename("count").reset_index()

def reindex_topic_platform(g, months_idx):
    g = g.set_index("month").reindex(months_idx)
    g["count"] = g["count"].fillna(0.0)
    total = g["count"].sum()
    g["norm"] = g["count"] / total if total > 0 else np.nan
    g = g.reset_index().rename(columns={"index":"month"})
    return g

# Build normalized series per topic-platform, then concatenate
pieces = []
for (t, p), g in grp.groupby(["topic", "platform"]):
    gg = reindex_topic_platform(g, months)
    gg["topic"] = t
    gg["platform"] = p
    pieces.append(gg)
ts_full = pd.concat(pieces, ignore_index=True)

# Plot per topic
topics = sorted(ts_full["topic"].unique(), key=lambda x: str(x))
saved = []
for t in topics:
    tdf = ts_full[ts_full["topic"] == t].copy()
    if tdf["norm"].notna().sum() == 0:
        continue
    plt.figure(figsize=(12, 4))
    for plat, g in tdf.groupby("platform"):
        x = g["month"].values
        y = g["norm"].values
        if np.all(np.isnan(y)):
            continue
        plt.plot(x, y, label=str(plat))
    plt.title(f"Topic {t} — normalized monthly share by platform (2010–2025)")
    plt.xlabel("Time")
    plt.ylabel("Normalized share within topic × platform")
    plt.legend()
    plt.tight_layout()
    out_path = os.path.join(OUT_DIR, f"topic_{str(t).replace(os.sep,'_')}.png")
    plt.savefig(out_path, dpi=150)
    plt.close()
    saved.append(out_path)

index_csv = os.path.join(OUT_DIR, "index_of_plots.csv")
pd.DataFrame({"plot_path": saved}).to_csv(index_csv, index=False)

print(f"Saved {len(saved)} topic plots to: {OUT_DIR}")
print(f"Index CSV: {index_csv}")

Sentiment over time

In [ ]:
# --- Canonical topic remap + monthly merged sentiment per topic × platform ---
# - Canonical mapping source: Reddit (topic_num -> topic name)
# - YouTube: uses 'topic' column; if numeric, treat as topic_num and map to canonical name
# - News/Reddit/PubMed: same logic; if topic_num present use it; otherwise use topic string
# - Date parsing is platform-aware to avoid dropping platforms
# - Sentiment computed AFTER merging all texts per (topic, platform, month)

import os, re, math, unicodedata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------- Config ----------------
DATA_DIR = TOPIC_MATCHINGS
PATHS = {
    "PubMed":  f"{DATA_DIR}/pubmed_with_topics.csv",
    "News":    f"{DATA_DIR}/news_with_topics.csv",
    "Reddit":  f"{DATA_DIR}/reddit_with_topics.csv",
    "YouTube": f"{DATA_DIR}/youtube_with_topics.csv",
}
OUT_DIR = "dont_use/topic_sentiment_timeseries_canonical"
DATE_START = "2010-01-01"
DATE_END   = "2025-12-31"
FREQ = "MS"  # monthly
os.makedirs(OUT_DIR, exist_ok=True)

# -------------- Utils -------------------
def normalize_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = unicodedata.normalize("NFKC", s).replace("\n"," ").replace("\r"," ").strip()
    return re.sub(r"\s+"," ", s)

def looks_numeric(series: pd.Series) -> pd.Series:
    # true if values are int/float strings like "13", "13.0"
    s = series.astype(str).str.strip()
    return s.str.match(r"^[+-]?\d+(\.\d+)?$")

def diag_text(df, platform, picked_cols, df_name="df"):
    # How many rows have any non-empty text across all object columns?
    obj_cols = [c for c in df.columns if df[c].dtype == "O"]
    any_text = df[obj_cols].fillna("").astype(str).agg(lambda r: any(len(x.strip())>0 for x in r), axis=1)
    # How many rows have non-empty in the picked columns?
    picked = [c for c in picked_cols if c in df.columns]
    if not picked:
        picked_nonempty = pd.Series([False]*len(df))
    else:
        picked_nonempty = df[picked].fillna("").astype(str).agg(lambda r: any(len(x.strip())>0 for x in r), axis=1)

    print(f"[{platform}] {df_name}: rows={len(df)} | obj_cols={len(obj_cols)} | picked_cols={picked}")
    print(f"[{platform}] any object-text present: {any_text.sum()} rows")
    print(f"[{platform}] picked-cols non-empty : {picked_nonempty.sum()} rows")
    # Show a few rows where object text exists but picked-cols are empty (bad selection)
    bad = (~picked_nonempty) & any_text
    if bad.any():
        print(f"[{platform}] WARNING: {bad.sum()} rows have text somewhere, but not in picked columns. Example rows:")
        print(df.loc[bad, obj_cols].head(3).T)

def choose_text_columns(df: pd.DataFrame, platform: str):
    prefer = {
        "Reddit":  ["title","selftext"],
        "YouTube": ["title","description","comments","keywords"],
        "PubMed":  ["title_abstract"],
        "News":    ["Title","Abstract"]
    }.get(platform, [])
    cols = [c for c in prefer if c in df.columns]
    if cols: return cols
    generic = ["title_abstract","text","content","body","abstract","description",
               "selftext","clean_text","full_text","title_and_abstract","title"]
    cols = [c for c in generic if c in df.columns]
    return cols if cols else [df.columns[0]]

def parse_date_platform(df: pd.DataFrame, platform: str) -> pd.Series:
    if platform == "PubMed" and "pub_date" in df.columns:
        return pd.to_datetime(df["pub_date"], errors="coerce")
    if platform == "News":
        if "Publication date" in df.columns:
            return pd.to_datetime(df["Publication date"], errors="coerce")
        if "Publication year" in df.columns:
            y = pd.to_numeric(df["Publication year"], errors="coerce")
            return pd.to_datetime(dict(year=y, month=1, day=1), errors="coerce")
    if platform == "Reddit":
        if "ts_seconds" in df.columns:
            return pd.to_datetime(df["ts_seconds"], unit="s", errors="coerce")
        if "ts_utc" in df.columns:
            return pd.to_datetime(df["ts_utc"], errors="coerce")
        if "created_utc" in df.columns:
            return pd.to_datetime(df["created_utc"], unit="s", errors="coerce")
    if platform == "YouTube":
        if "date" in df.columns:
            return pd.to_datetime(df["date"], errors="coerce")
        if "published_at" in df.columns:
            return pd.to_datetime(df["published_at"], errors="coerce")
    # fallbacks
    for c in ["date","Date","published_at","timestamp"]:
        if c in df.columns:
            try: return pd.to_datetime(df[c], errors="coerce")
            except: pass
    return pd.to_datetime(pd.Series([pd.NaT]*len(df)))

# ---------- Sentiment (VADER -> vaderSentiment -> offline) ----------
class OfflineLexiconSentiment:
    def __init__(self):
        pos = "good great excellent positive promising breakthrough success improvement effective beneficial robust safe hopeful optimistic innovative advance longevity youthful efficient helpful heal healthy resilient progress win supportive enhance"
        neg = "bad poor worse worst negative risk risky harm harmful danger dangerous failure unsafe concern concerning fear fearful alarming scandal crisis crisis-level uncertain controversy controversial doubt skeptical side-effect adverse aging decline degrade toxic"
        self.pos = set(pos.split()); self.neg = set(neg.split())
        self.negators = {"no","not","never","without","hardly","barely","rarely"}
        self.token_re = re.compile(r"[A-Za-z][A-Za-z'\-]+")
    def polarity(self, text: str) -> float:
        text = normalize_text(text).lower()
        toks = self.token_re.findall(text)
        if not toks: return 0.0
        score = 0.0; w=3
        for i,t in enumerate(toks):
            v = 1.0 if t in self.pos else (-1.0 if t in self.neg else 0.0)
            if v and any(u in self.negators for u in toks[max(0,i-w):i]): v = -v
            score += v
        score /= max(1.0, len(toks)**0.5)
        return float(max(-1.0, min(1.0, score/3.0)))

def get_sentiment():
    try:
        import nltk
        from nltk.sentiment import SentimentIntensityAnalyzer
        try:
            analyzer = SentimentIntensityAnalyzer()
        except LookupError:
            nltk.download("vader_lexicon", quiet=True)
            analyzer = SentimentIntensityAnalyzer()
        def fn(x: str) -> float:
            try: return float(analyzer.polarity_scores(normalize_text(x))["compound"])
            except: return 0.0
        return fn, "vader"
    except Exception:
        try:
            from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
            analyzer = SentimentIntensityAnalyzer()
            def fn(x: str) -> float:
                try: return float(analyzer.polarity_scores(normalize_text(x))["compound"])
                except: return 0.0
            return fn, "vaderSentiment"
        except Exception:
            return OfflineLexiconSentiment().polarity, "offline_lexicon"

sentiment_fn, backend = get_sentiment()
print("Sentiment backend used:", backend)

# ---------- Build canonical topic map from Reddit ----------
reddit = pd.read_csv(PATHS["Reddit"])
if not {"topic_num","topic"}.issubset(reddit.columns):
    raise RuntimeError("Reddit CSV must contain both 'topic_num' and 'topic' to build the mapping.")
topic_map = (reddit[["topic_num","topic"]]
             .dropna()
             .drop_duplicates()
             .set_index("topic_num")["topic"]
             .to_dict())
print(f"Canonical topic mapping from Reddit: {len(topic_map)} entries")

def canonical_topic_for_df(df: pd.DataFrame) -> pd.Series:
    """
    Map each row to canonical topic name:
    - If 'topic_num' exists: map via topic_map
    - Else if 'topic' exists and looks numeric: cast to int (or round float) and map
    - Else: use 'topic' as-is
    """
    if "topic_num" in df.columns:
        # robust cast to int-ish
        tnum = pd.to_numeric(df["topic_num"], errors="coerce")
        tnum = tnum.round().astype("Int64")
        name = tnum.map(topic_map)
        # fallback to raw topic if mapping missing
        if "topic" in df.columns:
            return name.fillna(df["topic"].astype(str))
        return name.astype(str)
    if "topic" in df.columns:
        raw = df["topic"].astype(str)
        # map numeric-like topics
        mask_num = looks_numeric(raw)
        tnum = pd.to_numeric(raw.where(mask_num), errors="coerce").round().astype("Int64")
        name = pd.Series(index=df.index, dtype="object")
        name.loc[mask_num] = tnum.map(topic_map)
        name.loc[~mask_num] = raw.loc[~mask_num]
        return name.fillna(raw).astype(str)
    # last resort: first object column
    obj = [c for c in df.columns if df[c].dtype == "O"]
    return (df[obj[0]].astype(str) if obj else pd.Series(["unknown_topic"]*len(df)))

# --------- Load all, remap topics, merge texts, compute monthly sentiment ---------
frames = []
for platform, path in PATHS.items():
    if not os.path.exists(path):
        print("Missing:", path); continue
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # Canonical topic label
    canon_topic = (canonical_topic_for_df(df)
                   .astype(str)
                   .str.replace(r"\s+"," ", regex=True)
                   .str.strip()
                   .str.lower()
                   .str.replace("_"," "))

    # Dates (platform-aware)
    dt = parse_date_platform(df, platform)

    # Texts
    cols = choose_text_columns(df, platform)
    cols = [c for c in cols if c in df.columns]
    text = df[cols].fillna("").astype(str).agg(" ".join, axis=1).map(normalize_text)

    tmp = pd.DataFrame({"platform": platform, "topic": canon_topic, "date": dt, "text": text})
    frames.append(tmp)

cols = choose_text_columns(df, platform)
diag_text(df, platform, cols, df_name=os.path.basename(path))

alldf = pd.concat(frames, ignore_index=True)

# Drop invalid/out-of-range dates and bin to months
mask = alldf["date"].notna() & (alldf["date"] >= DATE_START) & (alldf["date"] <= DATE_END)
alldf = alldf.loc[mask].copy()
alldf["month"] = alldf["date"].values.astype("datetime64[M]")

# Merge texts FIRST per (topic, platform, month)
merged = (alldf.groupby(["topic","platform","month"], observed=False)["text"]
          .agg(lambda s: " ".join(s))
          .reset_index())

# Sentiment on merged monthly text
def safe_score(s: str) -> float:
    try: return sentiment_fn(s)
    except: return 0.0

merged["sentiment"] = merged["text"].astype(str).map(safe_score)

# Build continuous month grid, keep pairs that appear at least once
months = pd.date_range(DATE_START, DATE_END, freq=FREQ)
pairs = merged[["topic","platform"]].drop_duplicates()
idx = pd.MultiIndex.from_product([pairs["topic"].unique(), pairs["platform"].unique(), months],
                                 names=["topic","platform","month"])
wide = merged.set_index(["topic","platform","month"])[["sentiment"]].reindex(idx)
mask_any = wide.groupby(level=["topic","platform"])["sentiment"].transform(lambda s: s.notna().any())
ts = wide[mask_any].reset_index()

# Save data + plots
ts_out = f"{OUT_DIR}/topic_platform_monthly_sentiment_canonical.csv"
ts.to_csv(ts_out, index=False)

saved = []
for t, g in ts.groupby("topic"):
    if not g["sentiment"].notna().any(): continue
    plt.figure(figsize=(12,4))
    for plat, gg in g.groupby("platform"):
        plt.plot(gg["month"].values, gg["sentiment"].values, label=str(plat))
    title = " ".join(str(t).split()).title()
    plt.title(f"Topic: {title} — monthly sentiment by platform (2010–2025)")
    plt.xlabel("Time"); plt.ylabel("Sentiment ([-1, 1])")
    plt.legend(); plt.tight_layout()
    out_path = f"{OUT_DIR}/topic_{str(t).replace(' ','_')}.png"
    plt.savefig(out_path, dpi=150); plt.close()
    saved.append(out_path)

print("Sentiment backend:", backend)
print("Saved CSV:", ts_out)
print(f"Saved {len(saved)} topic plots to {OUT_DIR}.")

In [ ]:
# ===== Diagnostics: platform-by-month counts + integrity checks =====
import os
import pandas as pd
import numpy as np

# -------- Config --------
DATA_DIR = TOPIC_MATCHINGS  # <-- adjust if needed
PATHS = {
    "PubMed":  f"{DATA_DIR}/pubmed_with_topics.csv",
    "News":    f"{DATA_DIR}/news_with_topics.csv",
    "Reddit":  f"{DATA_DIR}/reddit_with_topics.csv",
    "YouTube": f"{DATA_DIR}/youtube_with_topics.csv",
}
DATE_START = "2010-01-01"
DATE_END   = "2025-12-31"

# -------- Platform-aware date parsing --------
def parse_date_platform(df: pd.DataFrame, platform: str) -> pd.Series:
    if platform == "PubMed" and "pub_date" in df.columns:
        return pd.to_datetime(df["pub_date"], errors="coerce")
    if platform == "News":
        if "Publication date" in df.columns:
            return pd.to_datetime(df["Publication date"], errors="coerce")
        if "Publication year" in df.columns:
            y = pd.to_numeric(df["Publication year"], errors="coerce")
            return pd.to_datetime(dict(year=y, month=1, day=1), errors="coerce")
    if platform == "Reddit":
        if "ts_seconds" in df.columns:
            return pd.to_datetime(df["ts_seconds"], unit="s", errors="coerce")
        if "ts_utc" in df.columns:
            return pd.to_datetime(df["ts_utc"], errors="coerce")
        if "created_utc" in df.columns:
            return pd.to_datetime(df["created_utc"], unit="s", errors="coerce")
    if platform == "YouTube":
        if "date" in df.columns:
            return pd.to_datetime(df["date"], errors="coerce")
        if "published_at" in df.columns:
            return pd.to_datetime(df["published_at"], errors="coerce")
    # fallbacks
    for c in ["date","Date","published_at","timestamp"]:
        if c in df.columns:
            try:
                return pd.to_datetime(df[c], errors="coerce")
            except Exception:
                pass
    return pd.to_datetime(pd.Series([pd.NaT]*len(df)))

# -------- Load, parse dates, make month --------
frames = []
for plat, path in PATHS.items():
    if not os.path.exists(path):
        print(f"[WARN] Missing file for {plat}: {path}")
        continue
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    dt = parse_date_platform(df, plat)
    tmp = pd.DataFrame({"platform": plat, "date": dt})
    frames.append(tmp)

if not frames:
    raise RuntimeError("No inputs.")

all_dates = pd.concat(frames, ignore_index=True)
mask = all_dates["date"].notna() & (all_dates["date"] >= DATE_START) & (all_dates["date"] <= DATE_END)
all_dates = all_dates.loc[mask].copy()
all_dates["month"] = all_dates["date"].values.astype("datetime64[M]")

# -------- 1) Platform-by-month total counts (regardless of topic) --------
platform_month_counts = (
    all_dates.groupby(["platform", "month"]).size()
             .rename("n_posts")
             .reset_index()
             .sort_values(["platform","month"])
)

print("\n=== PLATFORM × MONTH: total posts/articles ===")
print(platform_month_counts.head(20).to_string(index=False))
print(f"... total rows: {len(platform_month_counts)}")
# Save if useful:
platform_month_counts.to_csv("platform_month_counts.csv", index=False)

# -------- 2) Topic × platform × month counts (requires topic col name guess) --------
# We'll try common topic columns; if none found in a file, that platform's topics are skipped for this table.
topic_frames = []
for plat, path in PATHS.items():
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    dt = parse_date_platform(df, plat)
    # pick a topic-like column
    topic_col = None
    for c in ["topic", "topic_label", "Topic", "topic_num", "topic_id", "dominant_topic", "topic_name"]:
        if c in df.columns:
            topic_col = c
            break
    if topic_col is None:
        # still allow platform x month counts; but can't do per-topic here
        continue
    tmp = pd.DataFrame({"platform": plat, "topic": df[topic_col].astype(str), "date": dt})
    tmp = tmp[tmp["date"].notna()]
    tmp["month"] = tmp["date"].values.astype("datetime64[M]")
    topic_frames.append(tmp[["platform","topic","month"]])

topic_month_counts = None
if topic_frames:
    topic_all = pd.concat(topic_frames, ignore_index=True)
    topic_month_counts = (
        topic_all.groupby(["topic","platform","month"]).size()
                 .rename("n_posts")
                 .reset_index()
    )
    print("\n=== TOPIC × PLATFORM × MONTH: counts (sample) ===")
    print(topic_month_counts.head(20).to_string(index=False))
    topic_month_counts.to_csv("topic_platform_month_counts.csv", index=False)
else:
    print("\n[NOTE] Could not find a topic column in at least one platform; skipped the topic-level table.")

# -------- 3) Optional: join with your sentiment time series to find anomalies --------
# If you already saved a CSV like 'topic_platform_monthly_sentiment_*.csv', point to it here:
SENT_TS = "topic_sentiment_timeseries_canonical/topic_platform_monthly_sentiment_canonical.csv"
if os.path.exists(SENT_TS):
    ts = pd.read_csv(SENT_TS, parse_dates=["month"])
    # How many rows have NaN sentiment?
    n_nan = ts["sentiment"].isna().sum()
    print(f"\n[Sentiment TS] Rows with NaN sentiment: {n_nan} / {len(ts)}")

    if topic_month_counts is not None:
        # Left-join counts → for any (topic,platform,month), check posts vs NaN sentiment
        j = (topic_month_counts.merge(ts, on=["topic","platform","month"], how="left"))
        bad = j[(j["n_posts"] > 0) & (j["sentiment"].isna())]
        print(f"[Integrity] Cases where posts>0 but sentiment is NaN: {len(bad)}")
        if len(bad):
            print(bad.head(20).to_string(index=False))
            bad.to_csv("anomalies_posts_but_nan_sentiment.csv", index=False)
else:
    print(f"\n[INFO] Sentiment TS file not found at: {SENT_TS}. Skipping anomaly check.")

Emotions

In [ ]:
# ========= Config: choose emotion backend =========
EMOTION_BACKEND = "goemotions"   # "goemotions" | "none"
GOEMOTIONS_MODEL = "joeddav/distilbert-base-uncased-go-emotions-student"  # compact & fast
EMO_BATCH_SIZE = 32
EMO_MAX_LEN = 192

USE_PER_PLATFORM_EMO_MODELS = False

EMO_MODELS = {
    "PubMed":  "allenai/biobert-base-cased-v1.1",             # or a custom fine-tune
    "News":    "j-hartmann/emotion-english-distilroberta-base",
    "Reddit":  "joeddav/distilbert-base-uncased-go-emotions-student",
    "YouTube": "joeddav/distilbert-base-uncased-go-emotions-student",
}

# ========= Emotion backends =========
class GoEmotions:
    """
    Returns probability distribution over GoEmotions labels per text.
    Output: list of dicts {label: prob, ...} per input, with the same label order across calls.
    """
    def __init__(self, model_name=GOEMOTIONS_MODEL, batch_size=EMO_BATCH_SIZE, max_len=EMO_MAX_LEN):
        import os, torch
        from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
        os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        # device: CUDA > MPS > CPU
        device = 0 if torch.cuda.is_available() else -1
        if device == -1 and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = torch.device("mps")
        self.pipe = pipeline("text-classification",
                             model=self.model, tokenizer=self.tokenizer,
                             return_all_scores=True, truncation=True, device=device)
        # stable label order
        self.labels = [self.model.config.id2label[i] for i in range(self.model.config.num_labels)]
        self.batch_size = batch_size
        self.max_len = max_len

    def probs(self, texts):
        """List[str] -> np.array shape (N, E) with probs per label (self.labels order)."""
        out = []
        for i in range(0, len(texts), self.batch_size):
            batch = [t if t else "" for t in texts[i:i+self.batch_size]]
            preds = self.pipe(batch, truncation=True, max_length=self.max_len, return_all_scores=True)
            for item in preds:
                # item is list of {'label': '...', 'score': p}
                lab2p = {d["label"]: float(d["score"]) for d in item}
                out.append([lab2p.get(lbl, 0.0) for lbl in self.labels])
        import numpy as np
        return np.array(out, dtype=np.float32)

EMO_MODEL_CACHE = {}

def get_emo_model_for_platform(platform: str):
    if EMOTION_BACKEND != "goemotions":
        return None
    model_name = EMO_MODELS.get(str(platform), GOEMOTIONS_MODEL)
    key = ("goemotions", model_name)
    if key not in EMO_MODEL_CACHE:
        EMO_MODEL_CACHE[key] = GoEmotions(model_name=model_name,
                                          batch_size=EMO_BATCH_SIZE,
                                          max_len=EMO_MAX_LEN)
        print(f"[Emotions] Loaded: {model_name} for {platform}")
    return EMO_MODEL_CACHE[key]

# Single global model (used when USE_PER_PLATFORM_EMO_MODELS=False)
def build_emotion_model():
    if EMOTION_BACKEND == "goemotions":
        print(f"Emotion backend: GoEmotions — {GOEMOTIONS_MODEL}")
        return GoEmotions(model_name=GOEMOTIONS_MODEL,
                          batch_size=EMO_BATCH_SIZE, max_len=EMO_MAX_LEN)
    print("Emotion backend: none")
    return None

# Initialize global model once
emo_model_global = build_emotion_model()

def score_emotions_with(model, text_series: pd.Series) -> pd.DataFrame:
    if model is None:
        return pd.DataFrame(index=text_series.index)
    texts = text_series.astype(str).tolist()
    probs = model.probs(texts)
    cols = [f"emo_{l}" for l in model.labels]
    return pd.DataFrame(probs, index=text_series.index, columns=cols)

def build_emotion_model():
    if EMOTION_BACKEND == "goemotions":
        print(f"Emotion backend: GoEmotions — {GOEMOTIONS_MODEL}")
        return GoEmotions(model_name=GOEMOTIONS_MODEL,
                          batch_size=EMO_BATCH_SIZE, max_len=EMO_MAX_LEN)
    print("Emotion backend: none")
    return None

emo_model = build_emotion_model()

def score_emotions(text_series: pd.Series) -> pd.DataFrame:
    """Return a DataFrame with one column per emotion label (probabilities summing ~1)."""
    if emo_model is None:
        return pd.DataFrame(index=text_series.index)
    texts = text_series.astype(str).tolist()
    probs = emo_model.probs(texts)  # (N, E)
    df = pd.DataFrame(probs, index=text_series.index, columns=[f"emo_{l}" for l in emo_model.labels])
    return df

Map topic names

In [ ]:
import pandas as pd

# Load the files
df_topics = pd.read_csv(TOPIC_MATCHINGS / "pubmed_with_topics.csv")
df_data = pd.read_csv(TOPIC_MATCHINGS / "pubmed_new_articles.csv")

# --- 1. Clean and Harmonize the Key Column ('topic_num') ---

# In the target file, clean up the float type to int and strip any hidden chars
df_data['topic_num'] = (
    df_data['topic_num'].astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
    .astype(int)
)

# In the source file, ensure 'topic_num' is clean integer
if df_topics['topic_num'].dtype == object:
    df_topics['topic_num'] = df_topics['topic_num'].str.strip()
df_topics['topic_num'] = df_topics['topic_num'].astype(int)

# --- 2. Guarantee Unique Keys in the Topic Source ---

# Drop duplicates based on 'topic_num', keeping the last one found.
# This resolves the InvalidIndexError and enforces the 1:1 rule.
df_topics_clean = df_topics.drop_duplicates(subset=['topic_num'], keep='last')

# --- 3. Create and Apply Map ---

# Create the mapping Series: {topic_num: topic_name}
topic_map = df_topics_clean.set_index('topic_num')['topic']

# Map the topic names onto the data DataFrame, replacing the existing 'topic' column
df_data['topic'] = df_data['topic_num'].map(topic_map)

# Save the updated DataFrame
output_file = 'pubmed_new_articles_with_mapped_topics.csv'
df_data.to_csv(output_file, index=False)

Sentiment over the years

In [ ]:
# --- Faster YEARLY sentiment per topic × platform ---
# Toggles:
PARALLEL_JOBS = -1          # use all cores; set to 1 to disable parallelism
USE_DOC_AVERAGE = True      # True = avg (length-weighted) per-doc sentiment; False = score merged mega-text
MAX_CHARS_PER_GROUP = 200_000  # only used if USE_DOC_AVERAGE=False (cap merged length)
RUN_DIAGNOSTICS = True     # set True only when debugging
DO_SENTIMENT = False        # set False to skip sentiment (for testing)
# ========= Config: choose sentiment backend =========
SENTIMENT_BACKEND = "roberta"   # "vader" | "roberta" | "offline"
RO_BERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
TRANSFORMERS_BATCH_SIZE = 32
TRANSFORMERS_MAX_LEN = 256      # truncation length per text; bump if needed
# New toggle to control how EMOTIONS are computed (independent of sentiment)
EMOTIONS_PER_DOC = True   # True => save per-doc emotions CSV; False => merged


import os, re, unicodedata, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from joblib import Parallel, delayed
except Exception:
    Parallel = None
    delayed = None
    PARALLEL_JOBS = 1

# ---------------- Config ----------------
DATA_DIR = TOPIC_MATCHINGS
PATHS = {
    "PubMed":  f"{DATA_DIR}/pubmed_new_articles_with_mapped_topics.csv",
    "News":    f"{DATA_DIR}/news_with_topics.csv",
    "Reddit":  f"{DATA_DIR}/reddit_with_topics.csv",
    "YouTube": f"{DATA_DIR}/youtube_final_data.csv",
}
OUT_DIR = "topic_sentiment_and_emotions_with_per_doc_emotions"
DATE_START = "2010-01-01"
DATE_END   = "2025-12-31"
YEARS = list(range(2010, 2026))
os.makedirs(OUT_DIR, exist_ok=True)

# -------------- Utils -------------------
def looks_numeric(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    return s.str.match(r"^[+-]?\d+(\.\d+)?$")

def diag_text(df, platform, picked_cols, df_name="df"):
    obj_cols = [c for c in df.columns if df[c].dtype == "O"]
    any_text = df[obj_cols].fillna("").astype(str).agg(lambda r: any(len(x.strip())>0 for x in r), axis=1)
    picked = [c for c in picked_cols if c in df.columns]
    picked_nonempty = df[picked].fillna("").astype(str).agg(lambda r: any(len(x.strip())>0 for x in r), axis=1) if picked else pd.Series([False]*len(df))
    print(f"[{platform}] {df_name}: rows={len(df)} | obj_cols={len(obj_cols)} | picked_cols={picked}")
    print(f"[{platform}] any object-text present: {any_text.sum()} rows")
    print(f"[{platform}] picked-cols non-empty : {picked_nonempty.sum()} rows")
    bad = (~picked_nonempty) & any_text
    if bad.any():
        print(f"[{platform}] WARNING: {bad.sum()} rows have text somewhere, but not in picked columns. Example rows:")
        print(df.loc[bad, obj_cols].head(3).T)

def choose_text_columns(df: pd.DataFrame, platform: str):
    #Rewrite these if needed
    prefer = {
        "Reddit":  ["title","selftext"],
        "YouTube": ["title","description","comments","keywords"],
        "PubMed":  ["title", "abstract"],
        "News":    ["Title","Abstract"]
    }.get(platform, [])
    cols = [c for c in prefer if c in df.columns]
    if cols: return cols
    generic = ["title_abstract","text","content","body","abstract","description",
               "selftext","clean_text","full_text","title_and_abstract","title", "abstract"]
    cols = [c for c in generic if c in df.columns]
    return cols if cols else [df.columns[0]]

def parse_date_platform(df: pd.DataFrame, platform: str) -> pd.Series:
    if platform == "PubMed":
        # Prefer normalized ISO 'date' if present
        if "date" in df.columns:
            return pd.to_datetime(df["date"], errors="coerce")
        # Fallback: parse dict-like strings in 'pub_date'
        if "pub_date" in df.columns:
            import ast
            def _coerce_pubdate(x):
                if pd.isna(x):
                    return pd.NaT
                s = str(x)
                # Handles strings like "{'Year': '2024', 'Month': 'Nov', 'Day': '26'}"
                if s.startswith("{") and "Year" in s:
                    try:
                        d = ast.literal_eval(s)
                        y = int(d.get("Year"))
                        m = d.get("Month", "Jan")
                        day = int(d.get("Day", 1))
                        return pd.to_datetime(f"{y}-{m}-{day}", errors="coerce")
                    except Exception:
                        return pd.NaT
                return pd.to_datetime(s, errors="coerce")
            return df["pub_date"].map(_coerce_pubdate)

    if platform == "News":
        if "Publication date" in df.columns:
            return pd.to_datetime(df["Publication date"], errors="coerce")
        if "Publication year" in df.columns:
            y = pd.to_numeric(df["Publication year"], errors="coerce")
            return pd.to_datetime(dict(year=y, month=1, day=1), errors="coerce")
    if platform == "Reddit":
        for c, unit in [("ts_seconds","s"), ("created_utc","s"), ("ts_utc",None)]:
            if c in df.columns:
                return pd.to_datetime(df[c], errors="coerce", unit=unit) if unit else pd.to_datetime(df[c], errors="coerce")
    if platform == "YouTube":
        for c in ["date","published_at"]:
            if c in df.columns:
                return pd.to_datetime(df[c], errors="coerce")
    for c in ["date","Date","published_at","timestamp"]:
        if c in df.columns:
            try: return pd.to_datetime(df[c], errors="coerce")
            except: pass
    return pd.to_datetime(pd.Series([pd.NaT]*len(df)))


# ========= Sentiment backends (pluggable) =========
import re, unicodedata, numpy as np

def normalize_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = unicodedata.normalize("NFKC", s).replace("\n"," ").replace("\r"," ").strip()
    return re.sub(r"\s+"," ", s)

class OfflineLexiconSentiment:
    def __init__(self):
        pos = "good great excellent positive promising breakthrough success improvement effective beneficial robust safe hopeful optimistic innovative advance longevity youthful efficient helpful heal healthy resilient progress win supportive enhance"
        neg = "bad poor worse worst negative risk risky harm harmful danger dangerous failure unsafe concern concerning fear fearful alarming scandal crisis crisis-level uncertain controversy controversial doubt skeptical side-effect adverse aging decline degrade toxic"
        self.pos = set(pos.split()); self.neg = set(neg.split())
        self.negators = {"no","not","never","without","hardly","barely","rarely"}
        self.token_re = re.compile(r"[A-Za-z][A-Za-z'\-]+")
    def score(self, texts):
        out = []
        for text in texts:
            text = normalize_text(text).lower()
            toks = self.token_re.findall(text)
            if not toks:
                out.append(0.0); continue
            score = 0.0; w=3
            for i,t in enumerate(toks):
                v = 1.0 if t in self.pos else (-1.0 if t in self.neg else 0.0)
                if v and any(u in self.negators for u in toks[max(0,i-w):i]): v = -v
                score += v
            score /= max(1.0, len(toks)**0.5)
            out.append(float(max(-1.0, min(1.0, score/3.0))))
        return out

class VaderSentiment:
    def __init__(self):
        try:
            import nltk
            from nltk.sentiment import SentimentIntensityAnalyzer
            try:
                self.analyzer = SentimentIntensityAnalyzer()
            except LookupError:
                nltk.download("vader_lexicon", quiet=True)
                self.analyzer = SentimentIntensityAnalyzer()
        except Exception:
            from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
            self.analyzer = SentimentIntensityAnalyzer()
    def score(self, texts):
        res = []
        for t in texts:
            try:
                res.append(float(self.analyzer.polarity_scores(normalize_text(t))["compound"]))
            except:
                res.append(0.0)
        return res

class RobertaSentiment:
    """
    Any HF pipeline model. Defaults to CardiffNLP (NEG/NEU/POS).
    We map model outputs to a single compound in [-1,1]:
      - For 3-class: compound = P(pos) - P(neg)
      - For 5-star:  map stars {1..5} -> linspace(-1,1) and take expectation
    """
    def __init__(self, model_name=RO_BERTA_MODEL, batch_size=TRANSFORMERS_BATCH_SIZE, max_len=TRANSFORMERS_MAX_LEN):
        import os, torch
        from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
        os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        # device: CUDA > MPS > CPU
        device = 0 if torch.cuda.is_available() else -1
        if device == -1 and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = torch.device("mps")
        self.pipe = pipeline("sentiment-analysis",
                             model=self.model, tokenizer=self.tokenizer,
                             truncation=True, return_all_scores=True, device=device)
        self.batch_size = batch_size
        self.max_len = max_len

    def _labels_to_score(self, item):
        # item can be either a dict with 'label' & 'score' (top-1) or list of dicts (all scores)
        if isinstance(item, list):
            lab2p = {d["label"].upper(): float(d["score"]) for d in item}
        else:
            # single dict => rebuild pseudo-distribution (less ideal). Prefer return_all_scores=True.
            lab2p = {str(item["label"]).upper(): float(item["score"])}

        labs = list(lab2p.keys())

        # 3-class mapping (NEG/NEU/POS style)
        if any("NEG" in l for l in labs) and any("POS" in l for l in labs):
            p_pos = sum(v for k,v in lab2p.items() if "POS" in k)
            p_neg = sum(v for k,v in lab2p.items() if "NEG" in k)
            comp = p_pos - p_neg
            return float(max(-1.0, min(1.0, comp)))

        # 5-star mapping
        star_keys = [k for k in labs if any(s in k for s in ["1", "2", "3", "4", "5", "STAR"])]
        if len(star_keys) >= 3:
            # try to pull numeric
            def to_star(k):
                m = re.search(r"(\d)", k)
                return int(m.group(1)) if m else None
            stars = {to_star(k): lab2p[k] for k in labs if to_star(k) is not None}
            if stars:
                # expectation over mapped values (-1..1)
                vals = {s: -1.0 + (s-1)*(2.0/4.0) for s in range(1,6)}  # 1→-1, 3→0, 5→1
                total = sum(stars.values())
                if total <= 0:
                    return 0.0
                exp = sum(vals[s]*p for s,p in stars.items())/total
                return float(max(-1.0, min(1.0, exp)))

        # fallback: if label contains POS/NEG only single prob
        lab = max(lab2p.items(), key=lambda x: x[1])[0]
        return 1.0 if "POS" in lab else (-1.0 if "NEG" in lab else 0.0)

    def score(self, texts):
        # batch with truncation
        out = []
        # use return_all_scores for stable mapping
        for i in range(0, len(texts), self.batch_size):
            batch = [t if t else "" for t in texts[i:i+self.batch_size]]
            preds = self.pipe(batch, truncation=True, max_length=self.max_len, return_all_scores=True)
            for item in preds:
                out.append(self._labels_to_score(item))
        return out

def build_sentiment_model():
    if SENTIMENT_BACKEND == "vader":
        print("Sentiment backend used: vader")
        return VaderSentiment()
    elif SENTIMENT_BACKEND == "roberta":
        print(f"Sentiment backend used: transformers — {RO_BERTA_MODEL}")
        return RobertaSentiment(model_name=RO_BERTA_MODEL,
                                batch_size=TRANSFORMERS_BATCH_SIZE,
                                max_len=TRANSFORMERS_MAX_LEN)
    else:
        print("Sentiment backend used: offline_lexicon")
        return OfflineLexiconSentiment()

sent_model = build_sentiment_model()

def score_texts(text_series: pd.Series) -> pd.Series:
    """Vectorized scoring wrapper that returns a float Series in [-1,1]."""
    texts = text_series.astype(str).tolist()
    try:
        scores = sent_model.score(texts)
    except Exception as e:
        # robust fallback to neutral if anything blows up mid-batch
        print("Sentiment scoring error, falling back to zeros:", e)
        scores = [0.0] * len(texts)
    return pd.Series(scores, index=text_series.index, dtype="float32")

# ---------- Canonical topic map from Reddit ----------
reddit = pd.read_csv(PATHS["Reddit"])
if not {"topic_num","topic"}.issubset(reddit.columns):
    raise RuntimeError("Reddit CSV must contain both 'topic_num' and 'topic'.")
topic_map = (reddit[["topic_num","topic"]]
             .dropna()
             .drop_duplicates()
             .set_index("topic_num")["topic"]
             .to_dict())
print(f"Canonical topic mapping from Reddit: {len(topic_map)} entries")

def canonical_topic_for_df(df: pd.DataFrame) -> pd.Series:
    if "topic_num" in df.columns:
        tnum = pd.to_numeric(df["topic_num"], errors="coerce").round().astype("Int64")
        name = tnum.map(topic_map)
        if "topic" in df.columns:
            return name.fillna(df["topic"].astype(str))
        return name.astype(str)
    if "topic" in df.columns:
        raw = df["topic"].astype(str)
        mask_num = looks_numeric(raw)
        tnum = pd.to_numeric(raw.where(mask_num), errors="coerce").round().astype("Int64")
        name = pd.Series(index=df.index, dtype="object")
        name.loc[mask_num] = tnum.map(topic_map)
        name.loc[~mask_num] = raw.loc[~mask_num]
        return name.fillna(raw).astype(str)
    obj = [c for c in df.columns if df[c].dtype == "O"]
    return (df[obj[0]].astype(str) if obj else pd.Series(["unknown_topic"]*len(df)))

# --------- Load, remap, and prepare per-document records ----------
frames = []
for platform, path in PATHS.items():
    if not os.path.exists(path):
        print("Missing:", path); continue
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    if RUN_DIAGNOSTICS:
        cols_dbg = choose_text_columns(df, platform)
        diag_text(df, platform, cols_dbg, df_name=os.path.basename(path))

    canon_topic = (canonical_topic_for_df(df)
                   .astype(str)
                   .str.replace(r"\s+"," ", regex=True)
                   .str.strip()
                   .str.lower()
                   .str.replace("_"," "))
    dt = parse_date_platform(df, platform)

    # Build text (fast-ish)
    cols = choose_text_columns(df, platform)
    cols = [c for c in cols if c in df.columns]
    # This is already one of the faster ways in pandas for multi-col concat
    text = df[cols].fillna("").astype(str).agg(" ".join, axis=1).map(normalize_text)

    tmp = pd.DataFrame({
        "platform": platform,
        "topic": canon_topic,
        "date": dt,
        "text": text
    })
    frames.append(tmp)

alldf = pd.concat(frames, ignore_index=True)

# Filter by date and extract year
mask = alldf["date"].notna() & (alldf["date"] >= DATE_START) & (alldf["date"] <= DATE_END)
alldf = alldf.loc[mask].copy()
alldf["year"] = alldf["date"].dt.year

# Reduce memory churn
alldf["platform"] = alldf["platform"].astype("category")
alldf["topic"] = alldf["topic"].astype("category")

alldf["doc_id"] = np.arange(len(alldf), dtype=np.int64)

# --- sanity check (recommended) ---
required_cols = {"platform","topic","date","year","text","doc_id"}
missing = required_cols - set(alldf.columns)
assert not missing, f"alldf missing columns: {missing}"

# ================= EMOTIONS (independent of sentiment path) =================
if EMOTIONS_PER_DOC:
    # score emotions per DOCUMENT
    emo_df = score_emotions_with(emo_model_global, alldf["text"])  # or per-platform model switch
    if not emo_df.empty:
        # attach metadata per doc
        emo_df["platform"] = alldf["platform"].values
        emo_df["year"] = alldf["year"].values
        emo_df["topic"] = alldf["topic"].values
        emo_df["doc_id"] = alldf["doc_id"].values

        # compute per-doc top emotion + prob + entropy
        emo_cols = [c for c in emo_df.columns if c.startswith("emo_")]
        probs = emo_df[emo_cols].to_numpy()
        top_idx = probs.argmax(axis=1)
        labels = [c.replace("emo_", "") for c in emo_cols]
        emo_df["top_emotion"] = [labels[i] for i in top_idx]
        emo_df["top_emotion_prob"] = probs[np.arange(len(probs)), top_idx].astype("float32")

        eps = 1e-9
        H = -(probs * np.log(probs + eps)).sum(axis=1)
        Hmax = np.log(len(emo_cols)) if emo_cols else 1.0
        emo_df["entropy"] = (H / Hmax).astype("float32")

        # save per-doc emotions (ridgeline-friendly)
        doc_out = f"{OUT_DIR}/topic_sentiment_highest_probability_emotions.csv"
        emo_df[["doc_id","platform","year","topic","top_emotion","top_emotion_prob","entropy"]].to_csv(doc_out, index=False)
        print("Saved per-document emotions:", doc_out)

        # optional: also build yearly length-weighted averages for plots
        if "doc_len" not in alldf.columns:
            alldf["doc_len"] = alldf["text"].str.len().clip(1)
        emo_df["w"] = alldf["doc_len"].astype("float32").values
        def wavg(g):
            w = g["w"].values.reshape(-1,1)
            X = g[emo_cols].values
            return pd.Series((X*w).sum(axis=0) / max(w.sum(), 1.0), index=emo_cols)
        emo_agg = (emo_df.groupby(["topic","platform","year"], observed=False).apply(wavg).reset_index())
    else:
        emo_agg = pd.DataFrame()
        print("Emotion backend produced no outputs (skipping emotion CSV).")
else:
    # score emotions on MERGED texts (one row per topic×platform×year)
    merged_for_emo = (alldf.groupby(["topic","platform","year"], observed=False)["text"]
                      .agg(lambda s: " ".join(s)).reset_index())
    if MAX_CHARS_PER_GROUP:
        merged_for_emo["text"] = merged_for_emo["text"].str.slice(0, MAX_CHARS_PER_GROUP)
    emo_df = score_emotions_with(emo_model_global, merged_for_emo["text"])
    if not emo_df.empty:
        emo_df[["topic","platform","year"]] = merged_for_emo[["topic","platform","year"]].values
        emo_agg = (emo_df.groupby(["topic","platform","year"], observed=False)
                   .mean(numeric_only=True).reset_index())
    else:
        emo_agg = pd.DataFrame()
        print("Emotion backend produced no outputs (skipping emotion CSV).")

# common post-process for emo_agg (top_emotion, top_prob, entropy) + save
if not emo_agg.empty:
    probs = emo_agg[[c for c in emo_agg.columns if c.startswith("emo_")]].values
    top_idx = probs.argmax(axis=1)
    emo_labels_clean = [c.replace("emo_","") for c in emo_agg.columns if c.startswith("emo_")]
    emo_agg["top_emotion"] = [emo_labels_clean[i] for i in top_idx]
    emo_agg["top_prob"] = probs[np.arange(len(probs)), top_idx]
    eps = 1e-9
    H = -(probs * np.log(probs + eps)).sum(axis=1)
    Hmax = np.log(probs.shape[1]) if probs.shape[1] else 1.0
    emo_agg["entropy"] = (H / Hmax).astype("float32")
    emo_out = f"{OUT_DIR}/topic_platform_yearly_emotions.csv"
    emo_agg.to_csv(emo_out, index=False)
    print("Saved emotions CSV:", emo_out)
# ================= END EMOTIONS =================

# --------- Build full year grid & save ----------
if DO_SENTIMENT:
    # --- existing sentiment computation (per-doc or merged) ---
    # produces `agg` with columns: topic, platform, year, sentiment

    # Build full year grid & save (sentiment)
    pairs = agg[["topic","platform"]].drop_duplicates()
    idx = pd.MultiIndex.from_product([pairs["topic"].unique(), pairs["platform"].unique(), YEARS],
                                     names=["topic","platform","year"])
    wide = agg.set_index(["topic","platform","year"])[["sentiment"]].reindex(idx)
    mask_any = wide.groupby(level=["topic","platform"])["sentiment"].transform(lambda s: s.notna().any())
    ts = wide[mask_any].reset_index()
    ts_out = f"{OUT_DIR}/topic_platform_yearly_sentiment_canonical.csv"
    ts.to_csv(ts_out, index=False)
    print("Saved CSV:", ts_out)

    # --------- Plots (each platform in its own subplot; one PNG per topic) ----------
    PLATFORM_ORDER = list(PATHS.keys())  # e.g., ["PubMed","News","Reddit","YouTube"]

    saved = []
    for t, g in ts.groupby("topic"):
        g = g.dropna(subset=["sentiment"])
        if g.empty:
            continue

        # Platforms that actually have data for this topic, in a stable order
        plats_present = [p for p in PLATFORM_ORDER if p in g["platform"].unique()]
        n = len(plats_present)
        if n == 0:
            continue

        # Layout: 1 col if 1 platform, else 2 cols; rows computed accordingly
        ncols = 1 if n == 1 else 2
        nrows = int(math.ceil(n / ncols))

        fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2 * nrows), sharex=True, sharey=True)
        axes = np.atleast_1d(axes).ravel()

        # Common y-limits for sentiment
        y_min, y_max = -1.0, 1.0

        for ax, plat in zip(axes, plats_present):
            gg = g[g["platform"] == plat].sort_values("year")
            ax.plot(gg["year"].values, gg["sentiment"].values, marker="o")
            ax.set_title(plat)
            ax.set_ylim(y_min, y_max)
            ax.grid(True, alpha=0.3)
            ax.set_xticks(YEARS[::2])

        # Hide any unused axes (when grid has extra slots)
        for ax in axes[len(plats_present):]:
            ax.set_visible(False)

        title = " ".join(str(t).split()).title()
        fig.suptitle(f"Topic: {title} — yearly sentiment by platform (2010–2025)", y=0.98, fontsize=12)
        # Shared labels
        fig.text(0.5, 0.04, "Year", ha="center")
        fig.text(0.04, 0.5, "Sentiment ([-1, 1])", va="center", rotation="vertical")

        plt.tight_layout(rect=[0.05, 0.06, 1, 0.96])
        out_path = f"{OUT_DIR}/topic_{str(t).replace(' ','_')}_yearly_by_platform.png"
        plt.savefig(out_path, dpi=150)
        plt.close(fig)
        saved.append(out_path)

    print("Saved subplot topic plots:", len(saved))

    print("Sentiment backend:", SENTIMENT_BACKEND, f"({type(sent_model).__name__})")
    print("Saved CSV:", ts_out)
    print(f"Saved {len(saved)} topic plots to {OUT_DIR}.")

    # --- existing plotting for sentiment (subplots per platform) ---
else:
    print("Sentiment disabled (DO_SENTIMENT=False). Skipping sentiment scoring/plots.")

In [ ]:
# Quick, on-machine benchmark
import time
sample_n = min(5000, len(alldf))   # try 5k first; raise if it feels fast
sample = alldf["text"].sample(sample_n, random_state=0)

t0 = time.time()
_ = score_texts(sample)            # uses your selected backend (e.g., RoBERTa)
dt = time.time() - t0

print(f"Scored {sample_n} docs in {dt:.2f}s  ->  {sample_n/dt:.1f} docs/sec")

Topic x year heatmap of sentiment per platform

In [ ]:
DATA_PATH = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data/topic_platform_yearly_sentiment_canonical.csv"
OUT_DIR   = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data/fig_sentiment_heatmaps"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import hashlib

# =================== Config ===================
DATA_PATH = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/topic_platform_yearly_sentiment_canonical.csv"
OUT_DIR   = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/fig_sentiment_heatmaps"
os.makedirs(OUT_DIR, exist_ok=True)

# visual style
sns.set_context("talk")
sns.set_style("whitegrid", {'axes.grid': False})

VMIN, VMAX = -1.0, 1.0
CMAP = "RdYlGn"
CELL_HEIGHT = 0.45
CELL_WIDTH  = 0.9
ADD_SENTIMENT_ANNOT = False

# NEW: label truncation
MAX_LABEL_CHARS = 36  # change as you like (visible characters)
MAPPING_OUT = os.path.join(OUT_DIR, "topic_label_mapping.csv")

def ellipsize(s: str, limit: int = 36) -> str:
    s = str(s)
    if len(s) <= limit:
        return s
    # keep whole words when possible; fall back to hard cut
    parts = s.split()
    if len(parts) == 1:  # single long token
        return s[:max(0, limit-1)] + "…"
    out = []
    for w in parts:
        candidate = (" ".join(out + [w])).strip()
        if len(candidate) <= limit - 1:  # keep room for ellipsis
            out.append(w)
        else:
            break
    if not out:  # very long first word
        return s[:max(0, limit-1)] + "…"
    return " ".join(out) + "…"

def make_unique_short_labels(full_labels, limit=36):
    """
    Returns list of short labels (ellipsized) that are unique.
    If collisions occur after truncation, append a tiny disambiguator.
    Also returns a mapping DataFrame (short -> full).
    """
    short = [ellipsize(lbl, limit) for lbl in full_labels]
    counts = {}
    resolved = []
    mapping_rows = []
    for full, sh in zip(full_labels, short):
        key = sh
        if key in counts:
            counts[key] += 1
            # disambiguate by appending a minimal suffix based on a short hash
            suffix = " " + hashlib.md5(full.encode("utf-8")).hexdigest()[:2]
            key_unique = (sh if len(sh) <= limit-3 else sh[:limit-3]) + "…" if not sh.endswith("…") else sh
            key_unique = (key_unique + suffix)
        else:
            counts[key] = 1
            key_unique = sh
        resolved.append(key_unique)
        mapping_rows.append({"short_label": key_unique, "full_label": full})
    mapping_df = pd.DataFrame(mapping_rows).drop_duplicates()
    return resolved, mapping_df

# =================== Load & aggregate (robust) ===================
df = pd.read_csv(DATA_PATH)

req = {"platform", "topic", "year", "sentiment"}
missing = req - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# 🔧 Force correct dtypes early
df["platform"]  = df["platform"].astype(str)
df["topic"]     = df["topic"].astype(str)
df["year"]      = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["sentiment"] = pd.to_numeric(df["sentiment"], errors="coerce")

df = df.dropna(subset=["platform", "topic", "year", "sentiment"]).copy()
df["year"] = df["year"].astype(int)

agg = (
    df.groupby(["platform", "topic", "year"], as_index=False)
      .agg(sentiment=("sentiment", "mean"),
           n=("sentiment", "size"))
)

all_years = np.sort(agg["year"].unique())

# =================== Plot per platform (robust) ===================
all_mappings = []
print("agg dtypes:", agg.dtypes.to_dict())
for platform, d in agg.groupby("platform", sort=False):
    # Build numeric pivot
    pivot = (
        d.pivot(index="topic", columns="year", values="sentiment")
         .reindex(columns=all_years)
         .sort_index()
         .apply(pd.to_numeric, errors="coerce")
         .astype(float)
    )

    # Build display labels
    full_topics = [str(x) for x in pivot.index.tolist()]
    short_labels, map_df = make_unique_short_labels(full_topics, limit=MAX_LABEL_CHARS)
    map_df.insert(0, "platform", platform)
    all_mappings.append(map_df)

    # --- Convert to plain ndarray + sanitize ---
    data = pivot.to_numpy(dtype=float)

    # Replace inf/-inf with nan (can happen after earlier transforms)
    data[~np.isfinite(data)] = np.nan

    # Optional: clip to expected sentiment bounds
    data = np.clip(data, VMIN, VMAX)

    # Guard: empty or all-nan case → skip plotting cleanly
    if data.size == 0 or not np.isfinite(data).any():
        print(f"⚠️ Skipping {platform}: no finite sentiment values to plot.")
        continue

    n_topics, n_years = data.shape
    # Guard: if either dimension is zero
    if n_topics == 0 or n_years == 0:
        print(f"⚠️ Skipping {platform}: empty pivot shape {data.shape}.")
        continue

    # Build mask with exact same shape
    mask = ~np.isfinite(data)

    # Figure size scales with content (adds label spacing)
    figsize = (max(6, n_years * CELL_WIDTH), max(6, n_topics * CELL_HEIGHT))
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=False)

    hm = sns.heatmap(
        data,
        ax=ax,
        mask=mask,
        cmap=CMAP,
        vmin=VMIN, vmax=VMAX,
        center=0.0,
        linewidths=0.6,
        linecolor="white",
        cbar_kws={"label": "Average sentiment", "shrink": 0.9, "pad": 0.02},
        xticklabels=list(pivot.columns.astype(str)),
        yticklabels=short_labels,
    )

    ax.set_title(f"Sentiment by Topic × Year — {platform}", pad=16, fontsize=16, weight="bold")
    ax.set_xlabel("Year", labelpad=10)
    ax.set_ylabel("Topic", labelpad=12)

    # Spacing/readability
    ax.tick_params(axis='y', which='major', labelsize=11, pad=14)
    ax.tick_params(axis='x', which='major', labelsize=11, rotation=0, pad=6)
    hm.figure.axes[-1].tick_params(labelsize=10)

    # Optional numeric annotations (OFF by default)
    if ADD_SENTIMENT_ANNOT:
        annot = np.where(np.isfinite(data), np.vectorize(lambda x: f"{x:+.2f}")(data), "")
        sns.heatmap(
            data,
            ax=ax,
            mask=mask,
            cmap=CMAP,
            vmin=VMIN, vmax=VMAX, center=0.0,
            annot=annot, fmt="",
            cbar=False,
            linewidths=0.6, linecolor="white",
            xticklabels=False, yticklabels=False  # keep ticks from the first heatmap
        )

    base = os.path.join(OUT_DIR, f"{platform}_sentiment_heatmap")
    plt.tight_layout()
    plt.savefig(base + ".png", dpi=300)
    plt.savefig(base + ".pdf")
    plt.close()

# save mapping once
if all_mappings:
    pd.concat(all_mappings, ignore_index=True).drop_duplicates().to_csv(MAPPING_OUT, index=False)

Basic emotion plots

In [ ]:
EMOS_TO_PLOT = ["joy","sadness","anger","fear","disgust","surprise","optimism","admiration","disappointment","neutral"]

if not emo_agg.empty:
    PLATFORM_ORDER = list(PATHS.keys())
    saved_emo = []
    for t, g in emo_agg.groupby("topic"):
        plats_present = [p for p in PLATFORM_ORDER if p in g["platform"].unique()]
        if not plats_present:
            continue
        n = len(plats_present)
        ncols = 1 if n == 1 else 2
        nrows = int(math.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.6*nrows), sharex=True, sharey=True)
        axes = np.atleast_1d(axes).ravel()

        # pick available emotions in this model
        emo_cols = [f"emo_{e}" for e in EMOS_TO_PLOT if f"emo_{e}" in g.columns]
        if not emo_cols:
            plt.close(fig); continue

        for ax, plat in zip(axes, plats_present):
            gg = g[g["platform"] == plat].sort_values("year")
            for ec in emo_cols:
                ax.plot(gg["year"].values, gg[ec].values, marker="o", label=ec.replace("emo_",""))
            ax.set_title(plat)
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            ax.set_xticks(YEARS[::2])

        for ax in axes[len(plats_present):]:
            ax.set_visible(False)

        title = " ".join(str(t).split()).title()
        fig.suptitle(f"Topic: {title} — yearly emotion probabilities by platform", y=0.98, fontsize=12)
        fig.text(0.5, 0.04, "Year", ha="center")
        fig.text(0.04, 0.5, "Probability", va="center", rotation="vertical")
        # one legend, outside
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc="upper center", ncol=min(5, len(labels)), frameon=False)
        plt.tight_layout(rect=[0.05, 0.10, 1, 0.95])
        out_path = f"{OUT_DIR}/topic_{str(t).replace(' ','_')}_yearly_emotions.png"
        plt.savefig(out_path, dpi=150); plt.close(fig)
        saved_emo.append(out_path)
    print("Saved emotion subplot topic plots:", len(saved_emo))

In [ ]:
import os
import pandas as pd
import numpy as np

EMO_CSV = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/topic_platform_yearly_emotions.csv"
OUT_DIR = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/top_emotions"
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(EMO_CSV)

# ---- wipe any index names and ensure flat index ----
if getattr(df.index, "names", [None]) != [None]:
    df = df.reset_index()
df.index = pd.RangeIndex(len(df), name=None)

# ---- rename the platform column to avoid any ambiguity with index names ----
if "platform" not in df.columns:
    raise RuntimeError(f"No 'platform' column found. Columns: {list(df.columns)}")

df = df.rename(columns={"platform": "_platform"})

# ---- emotion columns + cleaning ----
emo_cols = [c for c in df.columns if c.startswith("emo_")]
if not emo_cols:
    raise RuntimeError("No emotion probability columns found (columns starting with 'emo_').")

mask_allnan = df[emo_cols].isna().all(axis=1)
if mask_allnan.any():
    print("Rows with all emotion probabilities NaN:", int(mask_allnan.sum()))
    print(df.loc[mask_allnan, ["_platform","year","topic"]].head())

# keep rows with some emotion info AND a valid top_prob
dfv = (df.loc[~mask_allnan]
         .dropna(subset=["top_prob"])
         .copy())
dfv.index = pd.RangeIndex(len(dfv), name=None)

# 1) Overall winner per platform
overall = (dfv.sort_values(["_platform","top_prob"], ascending=[True, False])
             .drop_duplicates(subset=["_platform"], keep="first")
             [["_platform","year","topic","top_emotion","top_prob"]]
             .sort_values("_platform")
             .reset_index(drop=True))
print("\nOverall highest-probability emotion per platform:")
print(overall.to_string(index=False))
overall.to_csv(f"{OUT_DIR}/topic_sentiment_highest_probability_emotions.csv", index=False)

# 2) Winner per platform & year
winners_py = (dfv.sort_values(["_platform","year","top_prob"], ascending=[True, True, False])
                .drop_duplicates(subset=["_platform","year"], keep="first")
                [["_platform","year","topic","top_emotion","top_prob"]]
                .sort_values(["_platform","year"])
                .reset_index(drop=True))
print("\nHighest-probability emotion per platform per year:")
print(winners_py.to_string(index=False))
winners_py.to_csv(f"{OUT_DIR}/topic_sentiment_highest_probability_emotions_per_year.csv", index=False)

# 3) Emotion with highest average probability per platform
mean_probs = (dfv.groupby("_platform", as_index=False)[emo_cols]
                .mean(numeric_only=True)
                .set_index("_platform"))
top_mean = mean_probs.idxmax(axis=1).str.replace("emo_","")
top_mean_prob = mean_probs.max(axis=1)
avg_summary = (pd.DataFrame({
                    "platform": mean_probs.index,  # rename back for output
                    "emotion_with_highest_mean_prob": top_mean,
                    "mean_prob": top_mean_prob
               })
               .sort_values("platform")
               .reset_index(drop=True))
print("\nEmotion with highest average probability per platform:")
print(avg_summary.to_string(index=False))
avg_summary.to_csv(f"{OUT_DIR}/topic_sentiment_emotion_with_highest_average_probability_per_platform.csv", index=False)

# ---- 4) Top emotion per (topic, platform, year) ---------------------------
# We aggregate emotion probabilities within each (platform, year, topic)
# to be robust if there are multiple rows per combo (e.g., from different runs).

# keep only the columns we need for grouping + emotions
needed_cols = ["_platform", "year", "topic"] + emo_cols
df_group = dfv[needed_cols].copy()

# aggregate: mean of emotion probabilities per (platform, year, topic)
agg = (df_group
       .groupby(["_platform", "year", "topic"], as_index=False)[emo_cols]
       .mean(numeric_only=True))

# find the argmax emotion & its probability per group
agg["top_emotion"] = agg[emo_cols].idxmax(axis=1).str.replace("^emo_", "", regex=True)
agg["top_prob"] = agg[emo_cols].max(axis=1)

# nice output: rename _platform back to platform for the CSV
top_e_by_tpy = (agg
                .rename(columns={"_platform": "platform"})
                .loc[:, ["platform", "year", "topic", "top_emotion", "top_prob"]]
                .sort_values(["platform", "year", "topic"])
                .reset_index(drop=True))

print("\nTop emotion per (topic, platform, year):")
print(top_e_by_tpy.head(20).to_string(index=False))

out_path = f"{OUT_DIR}/top_emotion_per_topic_platform_year.csv"
top_e_by_tpy.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

Sectional emotion hist

In [ ]:
# Stacked bar charts (like the example image): one per platform, separately saved
# Each bar = a year, each stack section = an emotion (only those present that year/platform)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

# Load dataset
path = Path("topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/topic_platform_yearly_emotions.csv")
df = pd.read_csv(path)

# Normalize platform + year
def norm_platform(x: str) -> str:
    if not isinstance(x, str):
        return x
    y = x.strip().title()
    mapping = {"Youtube": "YouTube", "Reddit": "Reddit", "News": "News", "Pubmed": "PubMed", "X": "Twitter"}
    return mapping.get(y, y)

def to_year(x):
    try:
        if pd.isna(x):
            return np.nan
        m = re.search(r"\d{4}", str(x))
        return int(m.group()) if m else np.nan
    except Exception:
        return np.nan

df["platform"] = df["platform"].apply(norm_platform)
df["year"] = df["year"].apply(to_year)

# Emotion columns
emo_cols = [c for c in df.columns if c.lower().startswith("emo_")]
for c in emo_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Aggregate by platform-year
agg = df.groupby(["platform", "year"])[emo_cols].mean().reset_index()
agg = agg.dropna(subset=["platform", "year"])

# Output directory
out_dir = Path("topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/figures/stacked_bars")
out_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 200
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.autolayout"] = True

saved = []

# Create stacked bar chart per platform
for platform, g in agg.groupby("platform"):
    g = g.sort_values("year")
    g = g.dropna(subset=emo_cols, how="all")
    if g.empty:
        continue

    years = g["year"].astype(int).tolist()
    bottom = np.zeros(len(years))

    fig, ax = plt.subplots(figsize=(8, 5))
    for emo in emo_cols:
        vals = g[emo].fillna(0).values
        if np.any(vals > 0):
            ax.bar(years, vals, bottom=bottom, label=emo, width=0.7)
            bottom += vals

    ax.set_title(f"Emotion Distribution by Year — {platform}")
    ax.set_xlabel("Year")
    ax.set_ylabel("Average Emotion Probability")
    ax.set_ylim(0, 1.0)
    ax.legend(title="Emotions", bbox_to_anchor=(1.05, 1), loc="upper left", frameon=False)
    plt.tight_layout()

    out_path = out_dir / f"stacked_emotions_{platform.lower()}.png"
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    saved.append(out_path)

print(f"Saved {len(saved)} stacked bar charts:")
for s in saved:
    print("-", s)

Emotion Distribution Over Time per Platform

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

SAVE_DIR = f"topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/figures/pivot"
os.makedirs(SAVE_DIR, exist_ok=True)

df = pd.read_csv("topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/top_emotions/top_emotion_per_topic_platform_year.csv")

# Ensure clean types
df = df.dropna(subset=["platform", "year", "top_emotion"])
df["year"] = df["year"].astype(int).astype("int64")
df["top_emotion"] = df["top_emotion"].astype(str)

# Count how many times each emotion appears per (platform, year)
counts = (df.groupby(["platform","year","top_emotion"])
            .size()
            .reset_index(name="count"))

# ✅ Use transform to keep the same index as `counts`
den = counts.groupby(["platform","year"])["count"].transform("sum")
counts["share"] = counts["count"] / den

# Plot separately per platform
for platform, d in counts.groupby("platform", sort=True):
    pivot = (d.pivot(index="year", columns="top_emotion", values="share")
               .sort_index()
               .fillna(0.0))
    ax = pivot.plot(kind="area", stacked=True, figsize=(10,6))
    ax.set_title(f"Emotion distribution over years — {platform}")
    ax.set_ylabel("Share of top emotions")
    ax.set_xlabel("Year")
    ax.legend(title="Emotion", bbox_to_anchor=(1.02,1), loc="upper left")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/{platform}.png")
    plt.show()
    plt.close()

Emotion Distribution Over Time per Platform (per-doc)

In [ ]:
import matplotlib as mpl
from matplotlib.colors import to_hex
import numpy as np

In [ ]:
def make_distinct_palette(n, seed=7):
    """Build a long list of distinct, readable colors (color-blind friendly first)."""
    base_names = ["tab20", "tab20b", "tab20c", "Dark2", "Set1", "Set2", "Set3", "Paired", "Accent", "Pastel1"]
    colors = []
    for name in base_names:
        cmap = mpl.cm.get_cmap(name)
        # Many of these have a .colors list already; otherwise sample evenly.
        if hasattr(cmap, "colors"):
            colors.extend([to_hex(c) for c in cmap.colors])
        else:
            colors.extend([to_hex(cmap(i / (cmap.N - 1))) for i in range(cmap.N)])
    # de-dupe while preserving order
    seen = set()
    uniq = [c for c in colors if not (c in seen or seen.add(c))]
    # deterministically interleave to separate similar hues
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(uniq))
    uniq = [uniq[i] for i in order]
    return uniq[:n] if n <= len(uniq) else (uniq * ((n // len(uniq)) + 1))[:n]

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----- Fonts & export (readable on small figures) -----
BASE = 14  # bump up/down once; everything scales from here

plt.rcParams.update({
    "font.size": BASE,                # default text
    "axes.titlesize": BASE + 4,       # plot titles
    "axes.labelsize": BASE + 2,       # axis labels
    "xtick.labelsize": BASE + 1,      # tick labels
    "ytick.labelsize": BASE + 1,
    "legend.fontsize": BASE,          # legend text
    "legend.title_fontsize": BASE + 1,
    "figure.titlesize": BASE + 6,
    "savefig.dpi": 450,               # crisper PNGs for small displays
    "pdf.fonttype": 42,               # keep text as editable vectors in PDF
    "ps.fonttype": 42,
})

# Optional: slightly thicker default lines for clarity on small images
plt.rcParams["lines.linewidth"] = 1.2

# =================== Config ===================
PER_DOC_CSV = "topic_sentiment_and_emotions_with_per_doc_emotions/topic_sentiment_highest_probability_emotions.csv"
SAVE_DIR    = "topic_sentiment_and_emotions_with_per_doc_emotions/figures/pivot_perdoc"
os.makedirs(SAVE_DIR, exist_ok=True)

# Optional: limit to certain platforms or years
PLATFORMS_FILTER = None  # e.g., ["PubMed", "News", "YouTube", "Reddit"]
YEAR_MIN, YEAR_MAX = None, None  # e.g., 2010, 2025

# =================== Load & validate ===================
df = pd.read_csv(PER_DOC_CSV)

# Try to find an emotion column
emotion_col = None
for cand in ["emotion", "top_emotion", "pred_emotion", "label_emotion"]:
    if cand in df.columns:
        emotion_col = cand
        break
if emotion_col is None:
    raise ValueError("Could not find an emotion column (tried: 'emotion', 'top_emotion', 'pred_emotion', 'label_emotion').")

# Ensure required columns
required = ["platform", "year", emotion_col]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Clean types
df = df.dropna(subset=["platform", "year", emotion_col]).copy()
df["platform"] = df["platform"].astype(str)
df[emotion_col] = df[emotion_col].astype(str)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)

# Optional filters
if PLATFORMS_FILTER:
    df = df[df["platform"].isin(PLATFORMS_FILTER)].copy()
if YEAR_MIN is not None:
    df = df[df["year"] >= YEAR_MIN].copy()
if YEAR_MAX is not None:
    df = df[df["year"] <= YEAR_MAX].copy()

# If a weight/count column exists, use it; else weight=1 per doc
if "count" in df.columns:
    df["_w"] = pd.to_numeric(df["count"], errors="coerce").fillna(0.0)
else:
    df["_w"] = 1.0

# Keep a stable, global ordering of emotions across platforms
all_emotions = sorted(df[emotion_col].unique().tolist())

# =================== Aggregate counts and shares ===================
# Count docs per (platform, year, emotion)
counts = (
    df.groupby(["platform", "year", emotion_col], dropna=False)["_w"]
      .sum()
      .reset_index(name="count")
)

# Compute per-(platform, year) shares
den = counts.groupby(["platform", "year"])["count"].transform("sum").replace(0, np.nan)
counts["share"] = counts["count"] / den

# (Optional) save a tidy CSV of shares for later reuse
counts_out = os.path.join(SAVE_DIR, "per_platform_year_emotion_shares.csv")
counts.to_csv(counts_out, index=False)
print(f"📝 Saved shares: {counts_out}")

# Keep a stable, global ordering of emotions across platforms
all_emotions = sorted(df[emotion_col].unique().tolist())

# Fixed color per emotion (consistent in every figure)
_palette_list = make_distinct_palette(len(all_emotions))
EMOTION_COLOR = {emo: _palette_list[i] for i, emo in enumerate(all_emotions)}

# =================== Plot stacked area per platform ===================
for platform, d in counts.groupby("platform", sort=True):
    pivot = (
        d.pivot(index="year", columns=emotion_col, values="share")
         .reindex(columns=all_emotions)
         .sort_index()
         .fillna(0.0)
    )
    if not pivot.empty:
        yr_min, yr_max = pivot.index.min(), pivot.index.max()
        pivot = pivot.reindex(range(yr_min, yr_max + 1), fill_value=0.0)

    col_colors = [EMOTION_COLOR[e] for e in pivot.columns]

    # Use explicit fig/ax to control layout; constrained layout avoids cramped labels
    fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")
    pivot.plot(
        kind="area",
        stacked=True,
        ax=ax,
        color=col_colors,
        linewidth=0.9,
        alpha=0.95,
    )

    ax.set_title(f"Emotion distribution over years — {platform}", pad=10, fontweight="bold")
    ax.set_ylabel("Share of documents")
    ax.set_xlabel("Year")
    ax.set_ylim(0, 1)
    ax.margins(x=0)

    # (You can remove these since rcParams already sets sizes, but keeping is fine.)
    ax.tick_params(axis="x", labelsize=plt.rcParams["xtick.labelsize"])
    ax.tick_params(axis="y", labelsize=plt.rcParams["ytick.labelsize"])

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles, labels, title="Emotion",
        bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.,
        frameon=True, framealpha=0.9
    )

    # No need for tight_layout if using layout="constrained", but fine to keep:
    # plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"{platform}.png"))
    plt.savefig(os.path.join(SAVE_DIR, f"{platform}.pdf"))
    plt.close(fig)

Heatmaps of emotions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

SAVE_DIR = f"topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/figures/heatmap"
os.makedirs(SAVE_DIR, exist_ok=True)

df = pd.read_csv("topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/top_emotions/top_emotion_per_topic_platform_year.csv")
df = df.dropna(subset=["platform", "year", "top_emotion", "top_prob"]).copy()
df["year"] = df["year"].astype(int)

# 1) Count frequency per (platform, year, emotion)
freq = (df.groupby(["platform", "year", "top_emotion"])
          .size()
          .reset_index(name="count"))

# 2) Mean top_prob per (platform, year, emotion) — used as tie-breaker
mean_prob = (df.groupby(["platform", "year", "top_emotion"])["top_prob"]
               .mean()
               .reset_index(name="mean_top_prob"))

# 3) Combine and pick winner per (platform, year)
cand = freq.merge(mean_prob, on=["platform", "year", "top_emotion"], how="inner")

# Sort so the first row in each group is the winner: highest count, then highest mean_top_prob
cand_sorted = cand.sort_values(
    ["platform", "year", "count", "mean_top_prob"],
    ascending=[True, True, False, False]
)

winners = (cand_sorted
           .drop_duplicates(subset=["platform", "year"], keep="first")
           .sort_values(["platform", "year"])
           .reset_index(drop=True))

# Pivot for heatmap (cells contain the emotion string)
pivot = winners.pivot(index="platform", columns="year", values="top_emotion")

plt.figure(figsize=(12, 5))
# We annotate with emotion labels, but color by a categorical mapping to integers for a visible legend-free heatmap.
# Build a palette index for emotions:
emos = pd.Index(sorted(df["top_emotion"].unique()))
emo_to_code = {e: i for i, e in enumerate(emos)}
coded = pivot.replace(emo_to_code)

ax = sns.heatmap(
    coded,
    annot=pivot,  # show emotion text
    fmt="",
    cmap="viridis",
    cbar=False
)
ax.set_title("Most common top emotion per platform–year (ties → higher mean top_prob)")
ax.set_xlabel("Year")
ax.set_ylabel("Platform")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/heatmap.png")
plt.show()

Mean probability of top emotion: It can be used to measure certainness or agreement on the topics over the years.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

SAVE_DIR = f"topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/figures/emotion_prob"
os.makedirs(SAVE_DIR, exist_ok=True)

df = pd.read_csv("topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/top_emotions/top_emotion_per_topic_platform_year.csv")
df = df.dropna(subset=["platform", "year", "top_prob"]).copy()
df["year"] = df["year"].astype(int)

avgprob = (df.groupby(["platform", "year"])["top_prob"]
             .mean()
             .reset_index()
             .sort_values(["platform", "year"]))

plt.figure(figsize=(10, 6))
for platform, d in avgprob.groupby("platform", sort=True):
    d = d.sort_values("year")
    plt.plot(d["year"], d["top_prob"], marker="o", label=platform)

plt.title("Average top-emotion probability over time")
plt.ylabel("Mean probability of top emotion")
plt.xlabel("Year")
plt.grid()
plt.legend(title="Platform", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/emotion_prob.png")
plt.show()

Emotional certainty (per-doc)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# =================== Config ===================
PER_DOC_CSV = "topic_sentiment_and_emotions_with_per_doc_emotions/topic_sentiment_highest_probability_emotions.csv"
SAVE_DIR = "topic_sentiment_and_emotions_with_per_doc_emotions/figures/emotion_prob_perdoc"
os.makedirs(SAVE_DIR, exist_ok=True)

# =================== Load & clean ===================
df = pd.read_csv(PER_DOC_CSV)
if "top_emotion_prob" not in df.columns:
    raise ValueError("Per-doc CSV must include a 'top_emotion_prob' column for this plot.")

df = df.dropna(subset=["platform", "year", "top_emotion_prob"]).copy()
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)
df["platform"] = df["platform"].astype(str)

# =================== Compute yearly means ===================
avgprob = (
    df.groupby(["platform", "year"])["top_emotion_prob"]
      .mean()
      .reset_index()
      .sort_values(["platform", "year"])
)

# Determine global min/max for auto-zoom with margin
y_min = avgprob["top_emotion_prob"].min()
y_max = avgprob["top_emotion_prob"].max()
margin = (y_max - y_min) * 0.25  # add 25% margin
y_lower = max(0, y_min - margin)
y_upper = min(1, y_max + margin)

# =================== Plot ===================
plt.figure(figsize=(12, 8))
for platform, d in avgprob.groupby("platform", sort=True):
    d = d.sort_values("year")
    plt.plot(d["year"], d["top_emotion_prob"], marker="o", linewidth=2, label=platform)

plt.title("Average top-emotion probability over time", fontsize=15, weight="bold")
plt.ylabel("Mean probability of top emotion")
plt.xlabel("Year")
plt.ylim(y_lower, y_upper)  # zoomed-in range
plt.grid(alpha=0.3)
plt.legend(title="Platform", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/emotion_prob_zoomed.png", dpi=300)
plt.savefig(f"{SAVE_DIR}/emotion_prob_zoomed.pdf")
plt.show()
plt.close()

Heatmap of emotion counts per topic

In [ ]:
import os, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =================== Config ===================
DATA_PATH = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/top_emotions/top_emotion_per_topic_platform_year.csv"
SAVE_DIR  = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/figures/emotion_matrix"
os.makedirs(SAVE_DIR, exist_ok=True)

# Visual style
sns.set_context("talk")  # larger fonts
sns.set_style("whitegrid", {'axes.grid': False})

# Heatmap appearance
CMAP = "YlOrBr"          # sequential for counts; try "mako"/"crest" for cooler look
CELL_HEIGHT = 0.40        # vertical spacing between topics (increase for more padding)
CELL_WIDTH  = 0.55        # horizontal spacing between emotions

# What to display in cells
ADD_ANNOT = False         # True -> show counts in each cell
ANNOT_FMT = "{:,.0f}"

# Topic selection
TOP_N = 40                # keep the most “active” topics per platform

# Label truncation
MAX_LABEL_CHARS = 36
MAPPING_OUT = os.path.join(SAVE_DIR, "topic_label_mapping.csv")

# Normalization option for the heatmap values:
#   "none" (raw counts), "row" (per-topic shares), "col" (per-emotion shares)
NORMALIZE = "none"

# Optional transform to compress heavy tails: None, "log1p", or "sqrt"
VALUE_TRANSFORM = None

# =================== Helpers ===================
def ellipsize(s: str, limit: int = 36) -> str:
    s = str(s)
    if len(s) <= limit:
        return s
    parts = s.split()
    if len(parts) == 1:
        return s[:max(0, limit-1)] + "…"
    out = []
    for w in parts:
        candidate = (" ".join(out + [w])).strip()
        if len(candidate) <= limit - 1:
            out.append(w)
        else:
            break
    if not out:
        return s[:max(0, limit-1)] + "…"
    return " ".join(out) + "…"

def make_unique_short_labels(full_labels, limit=36):
    short = [ellipsize(lbl, limit) for lbl in full_labels]
    counts = {}
    resolved = []
    mapping_rows = []
    for full, sh in zip(full_labels, short):
        if sh in counts:
            counts[sh] += 1
            suffix = " " + hashlib.md5(full.encode("utf-8")).hexdigest()[:2]
            key_unique = (sh if sh.endswith("…") else (sh if len(sh) <= limit-3 else sh[:limit-3] + "…"))
            key_unique = key_unique + suffix
        else:
            counts[sh] = 1
            key_unique = sh
        resolved.append(key_unique)
        mapping_rows.append({"short_label": key_unique, "full_label": full})
    return resolved, pd.DataFrame(mapping_rows).drop_duplicates()

def normalize_matrix(df_counts, how="none"):
    if how == "row":
        denom = df_counts.sum(axis=1).replace(0, np.nan)
        return df_counts.div(denom, axis=0)
    if how == "col":
        denom = df_counts.sum(axis=0).replace(0, np.nan)
        return df_counts.div(denom, axis=1)
    return df_counts

def transform_values(a, mode=None):
    if mode is None:
        return a
    if mode == "log1p":
        return np.log1p(a)
    if mode == "sqrt":
        return np.sqrt(a)
    return a

# =================== Load & clean ===================
df = pd.read_csv(DATA_PATH)

# force reliable dtypes
for col in ["platform", "topic", "top_emotion"]:
    df[col] = df[col].astype(str)
# year may not be present here, but it doesn't matter for this aggregate
# Drop rows missing essentials
df = df.dropna(subset=["platform", "topic", "top_emotion"]).copy()

# =================== Collect all emotion categories (consistent across plots) ===================
all_emotions = sorted(df["top_emotion"].unique().tolist())

# Keep a global label mapping we’ll save once
all_maps = []

# =================== Per-platform plotting ===================
for platform, d in df.groupby("platform", sort=True):
    # Count occurrences per (topic, top_emotion)
    topic_emo = (
        d.groupby(["topic", "top_emotion"])
         .size()
         .reset_index(name="count")
    )

    # Pick top-N topics by total counts
    top_topics = (
        topic_emo.groupby("topic")["count"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(TOP_N)
                 .index
    )

    d_top = topic_emo[topic_emo["topic"].isin(top_topics)].copy()

    # Pivot to Topic × Emotion matrix (counts)
    pivot = (
        d_top.pivot(index="topic", columns="top_emotion", values="count")
             .reindex(columns=all_emotions)    # consistent emotion order across platforms
             .fillna(0)
    )

    # Sort topics by total (desc)
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    # Build display labels (short) + mapping
    full_topics = pivot.index.astype(str).tolist()
    short_labels, map_df = make_unique_short_labels(full_topics, limit=MAX_LABEL_CHARS)
    map_df.insert(0, "platform", platform)
    all_maps.append(map_df)

    # Prepare numeric matrix
    values = pivot.astype(float).to_numpy(dtype=float)
    values[~np.isfinite(values)] = np.nan

    # Normalize (optional)
    if NORMALIZE != "none":
        norm_df = normalize_matrix(pivot.astype(float), how=NORMALIZE)
        values = norm_df.to_numpy(dtype=float)

    # Transform values (optional)
    values = transform_values(values, VALUE_TRANSFORM)

    # Guard: skip if empty or all NaN
    if values.size == 0 or not np.isfinite(values).any():
        print(f"⚠️ Skipping {platform}: nothing to plot.")
        continue

    n_topics, n_emotions = values.shape
    mask = ~np.isfinite(values)

    # Figure size scales with data → comfortable label spacing
    figsize = (max(6, n_emotions * CELL_WIDTH), max(6, n_topics * CELL_HEIGHT))
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=False)

    hm = sns.heatmap(
        values,
        ax=ax,
        mask=mask,
        cmap=CMAP,
        linewidths=0.6,
        linecolor="white",
        cbar_kws={
            "label": "Count" if NORMALIZE == "none" else ("Share per topic" if NORMALIZE == "row" else "Share per emotion"),
            "shrink": 0.9,
            "pad": 0.02
        },
        xticklabels=pivot.columns.tolist(),
        yticklabels=short_labels,
    )

    # Titles & labels
    title_unit = "counts"
    if NORMALIZE == "row":
        title_unit = "row-normalized share"
    elif NORMALIZE == "col":
        title_unit = "column-normalized share"

    ax.set_title(f"Topic–Emotion intensity — {platform} (Top {len(pivot)} topics)", pad=16, fontsize=16, weight="bold")
    ax.set_xlabel("Emotion", labelpad=10)
    ax.set_ylabel("Topic", labelpad=12)

    # Tick styling & padding
    # (rotate emotions 90° to avoid overlap)
    ax.tick_params(axis='y', which='major', labelsize=11, pad=14)
    ax.tick_params(axis='x', which='major', labelsize=11, pad=8)  # don't set rotation here
    plt.setp(ax.get_xticklabels(), rotation=90, ha='center', va='center_baseline')

    # If labels still clip at the bottom, add a bit more bottom margin:
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.18)  # tweak 0.15–0.25 as needed

    # Optional numeric annotations (off by default)
    if ADD_ANNOT:
        annot_matrix = np.where(np.isfinite(values), values, np.nan)
        # choose format
        fmt_func = (lambda x: ANNOT_FMT.format(x)) if NORMALIZE == "none" else (lambda x: f"{x:.2f}")
        annot_text = np.empty_like(annot_matrix, dtype=object)
        for i in range(annot_matrix.shape[0]):
            for j in range(annot_matrix.shape[1]):
                annot_text[i, j] = "" if not np.isfinite(annot_matrix[i, j]) else fmt_func(annot_matrix[i, j])

        sns.heatmap(
            values,
            ax=ax,
            mask=mask,
            cmap=CMAP,
            annot=annot_text,
            fmt="",
            cbar=False,
            linewidths=0.6,
            linecolor="white",
            xticklabels=False,
            yticklabels=False
        )

    # Save PNG (300dpi) + PDF (vector)
    base = os.path.join(SAVE_DIR, f"{platform}_emotion_matrix")
    plt.tight_layout()
    plt.savefig(base + ".png", dpi=300)
    plt.savefig(base + ".pdf")
    plt.close()
    print(f"✅ Saved: {base}.png and {base}.pdf")

# Save label mapping once
if all_maps:
    pd.concat(all_maps, ignore_index=True).drop_duplicates().to_csv(MAPPING_OUT, index=False)
    print(f"📝 Saved label mapping: {MAPPING_OUT}")

Heatmap of emotion counts (per-doc) for each topic

In [ ]:
import os, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =================== Config ===================
# 👉 Use a PER-DOCUMENT file here: one row = one document (or a pre-aggregated row with a `count` column)
DATA_PATH = "topic_sentiment_and_emotions_with_per_doc_emotions/topic_sentiment_highest_probability_emotions.csv"
SAVE_DIR  = "topic_sentiment_and_emotions_with_per_doc_emotions/figures/emotion_matrix"
os.makedirs(SAVE_DIR, exist_ok=True)

# Visual style
sns.set_context("talk")
sns.set_style("whitegrid", {'axes.grid': False})

# Heatmap appearance
CMAP = "YlOrBr"
CELL_HEIGHT = 0.40
CELL_WIDTH  = 0.55

# What to display in cells
ADD_ANNOT = False
ANNOT_FMT = "{:,.0f}"

# Topic selection
TOP_N = 40

# Label truncation
MAX_LABEL_CHARS = 36
MAPPING_OUT = os.path.join(SAVE_DIR, "topic_label_mapping.csv")

# Always row-normalize (share per topic)
NORMALIZE = "row"

# Optional transform for raw counts before normalization (None recommended)
VALUE_TRANSFORM = None

# =================== Helpers ===================
def ellipsize(s: str, limit: int = 36) -> str:
    s = str(s)
    if len(s) <= limit:
        return s
    parts = s.split()
    if len(parts) == 1:
        return s[:max(0, limit-1)] + "…"
    out = []
    for w in parts:
        candidate = (" ".join(out + [w])).strip()
        if len(candidate) <= limit - 1:
            out.append(w)
        else:
            break
    if not out:
        return s[:max(0, limit-1)] + "…"
    return " ".join(out) + "…"

def make_unique_short_labels(full_labels, limit=36):
    short = [ellipsize(lbl, limit) for lbl in full_labels]
    counts = {}
    resolved = []
    mapping_rows = []
    for full, sh in zip(full_labels, short):
        if sh in counts:
            counts[sh] += 1
            suffix = " " + hashlib.md5(full.encode("utf-8")).hexdigest()[:2]
            key_unique = (sh if sh.endswith("…") else (sh if len(sh) <= limit-3 else sh[:limit-3] + "…"))
            key_unique = key_unique + suffix
        else:
            counts[sh] = 1
            key_unique = sh
        resolved.append(key_unique)
        mapping_rows.append({"short_label": key_unique, "full_label": full})
    return resolved, pd.DataFrame(mapping_rows).drop_duplicates()

def normalize_matrix(df_counts, how="row"):
    if how == "row":
        denom = df_counts.sum(axis=1).replace(0, np.nan)
        return df_counts.div(denom, axis=0)
    if how == "col":
        denom = df_counts.sum(axis=0).replace(0, np.nan)
        return df_counts.div(denom, axis=1)
    return df_counts

def transform_values(a, mode=None):
    if mode is None:
        return a
    if mode == "log1p":
        return np.log1p(a)
    if mode == "sqrt":
        return np.sqrt(a)
    return a

# =================== Load & validate ===================
df_raw = pd.read_csv(DATA_PATH)

# Try to find the emotion column
emotion_col = None
for cand in ["emotion", "top_emotion", "pred_emotion", "label_emotion"]:
    if cand in df_raw.columns:
        emotion_col = cand
        break

required_base = ["platform", "topic"]
missing = [c for c in required_base if c not in df_raw.columns]
if missing or (emotion_col is None):
    raise ValueError(
        f"Input must contain columns {required_base} and an emotion column "
        f"(one of: 'emotion', 'top_emotion', 'pred_emotion', 'label_emotion'). "
        f"Missing: {missing}; emotion_col={emotion_col}"
    )

# If neither doc_id nor count exist, and there IS a 'year' column,
# it's likely a 'top emotion per topic-year' table — not suitable for counting docs.
has_doc_id = any(c in df_raw.columns for c in ["doc_id", "document_id", "id"])
has_count  = ("count" in df_raw.columns)
if (not has_doc_id) and (not has_count) and ("year" in df_raw.columns):
    raise ValueError(
        "This looks like a per-year top-emotion table (has 'year' but no 'doc_id'/'count'). "
        "You can’t get document counts from that. Use a per-document file or provide a 'count' column."
    )

# Normalize dtypes
df_raw["platform"] = df_raw["platform"].astype(str)
df_raw["topic"]    = df_raw["topic"].astype(str)
df_raw[emotion_col]= df_raw[emotion_col].astype(str)

# If a weight/count column exists, use it; else every row = 1 document
if has_count:
    df_raw["_weight"] = pd.to_numeric(df_raw["count"], errors="coerce").fillna(0.0)
else:
    df_raw["_weight"] = 1.0

# Collect all emotions (stable order across platforms)
all_emotions = sorted(df_raw[emotion_col].unique().tolist())

all_maps = []

# =================== Per-platform plotting ===================
for platform, d in df_raw.groupby("platform", sort=True):
    # Aggregate total documents per (topic, emotion) using weights if present
    topic_emo = (
        d.groupby(["topic", emotion_col], dropna=False)["_weight"]
         .sum()
         .reset_index(name="count")
    )

    # Select top topics by total doc count
    top_topics = (
        topic_emo.groupby("topic")["count"]
                 .sum()
                 .sort_values(ascending=False)
                 .head(TOP_N)
                 .index
    )
    d_top = topic_emo[topic_emo["topic"].isin(top_topics)].copy()

    # Pivot to Topic × Emotion
    pivot = (
        d_top.pivot(index="topic", columns=emotion_col, values="count")
             .reindex(columns=all_emotions)
             .fillna(0.0)
    )

    # Sort topics by total docs desc
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    # Short labels + mapping
    full_topics = pivot.index.astype(str).tolist()
    short_labels, map_df = make_unique_short_labels(full_topics, limit=MAX_LABEL_CHARS)
    map_df.insert(0, "platform", platform)
    all_maps.append(map_df)

    # Optionally transform raw counts before normalization
    pre_vals = pivot.astype(float).to_numpy(dtype=float)
    pre_vals = transform_values(pre_vals, VALUE_TRANSFORM)
    pivot_transformed = pd.DataFrame(pre_vals, index=pivot.index, columns=pivot.columns)

    # Row-normalize to get per-topic shares
    norm_df = normalize_matrix(pivot_transformed, how="row")
    values = norm_df.to_numpy(dtype=float)
    mask = ~np.isfinite(values)

    if values.size == 0 or not np.isfinite(values).any():
        print(f"⚠️ Skipping {platform}: nothing to plot.")
        continue

    n_topics, n_emotions = values.shape
    figsize = (max(6, n_emotions * CELL_WIDTH), max(6, n_topics * CELL_HEIGHT))
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=False)

    hm = sns.heatmap(
        values,
        ax=ax,
        mask=mask,
        cmap=CMAP,
        linewidths=0.6,
        linecolor="white",
        cbar_kws={"label": "Share per topic", "shrink": 0.9, "pad": 0.02},
        xticklabels=pivot.columns.tolist(),
        yticklabels=short_labels,
        vmin=0.0, vmax=1.0
    )

    ax.set_title(f"Topic–Emotion share (row-normalized) — {platform} (Top {len(pivot)} topics)",
                 pad=16, fontsize=16, weight="bold")
    ax.set_xlabel("Emotion", labelpad=10)
    ax.set_ylabel("Topic", labelpad=12)
    ax.tick_params(axis='y', which='major', labelsize=11, pad=14)
    ax.tick_params(axis='x', which='major', labelsize=11, pad=8)
    plt.setp(ax.get_xticklabels(), rotation=90, ha='center', va='center_baseline')

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.18)

    if ADD_ANNOT:
        annot_matrix = np.where(np.isfinite(values), values, np.nan)
        annot_text = np.empty_like(annot_matrix, dtype=object)
        for i in range(annot_matrix.shape[0]):
            for j in range(annot_matrix.shape[1]):
                annot_text[i, j] = "" if not np.isfinite(annot_matrix[i, j]) else f"{annot_matrix[i, j]:.2f}"
        sns.heatmap(
            values,
            ax=ax,
            mask=mask,
            cmap=CMAP,
            annot=annot_text,
            fmt="",
            cbar=False,
            linewidths=0.6,
            linecolor="white",
            xticklabels=False,
            yticklabels=False
        )

    base = os.path.join(SAVE_DIR, f"{platform}_emotion_matrix_rowshare")
    plt.tight_layout()
    plt.savefig(base + ".png", dpi=300)
    plt.savefig(base + ".pdf")
    plt.close()
    print(f"✅ Saved: {base}.png and {base}.pdf")

# Save label mapping once
if all_maps:
    pd.concat(all_maps, ignore_index=True).drop_duplicates().to_csv(MAPPING_OUT, index=False)
    print(f"📝 Saved label mapping: {MAPPING_OUT}")

Grouped Bars

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

CSV = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/top_emotions/top_emotion_per_topic_platform_year.csv"
SAVE_DIR  = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/figures/grouped_bars"
os.makedirs(SAVE_DIR, exist_ok=True)
#TARGET_YEAR = [2017, 2018, 2019]
#TARGET_YEAR = [2020, 2021, 2022]
TARGET_YEAR = [2023, 2024, 2025]
#TARGET_YEAR = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]  # set e.g. 2022; if None, auto-choose the latest year present

df = pd.read_csv(CSV)
df = df.dropna(subset=["platform", "year", "top_emotion"]).copy()
df["year"] = df["year"].astype(int)

if TARGET_YEAR is None:
    TARGET_YEAR = int(df["year"].max())

subset = df[df["year"].isin(TARGET_YEAR)].copy()

counts = (subset.groupby(["platform", "top_emotion"])
                .size()
                .reset_index(name="count"))

# Normalize to shares per platform
den = counts.groupby("platform")["count"].transform("sum")
counts["share"] = counts["count"] / den

# Pivot to platforms × emotions
pivot = counts.pivot(index="platform", columns="top_emotion", values="share").fillna(0.0)
pivot = pivot.reindex(sorted(pivot.columns), axis=1).sort_index()

# Plot grouped bars
ax = pivot.plot(kind="bar", figsize=(12, 6))
ax.set_title(f"Emotion shares by platform — {TARGET_YEAR}")
ax.set_ylabel("Share of top emotions")
ax.set_xlabel("Platform")
ax.legend(title="Emotion", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.savefig(f"{SAVE_DIR}/grouped_emotion_bars_{'_'.join(map(str,TARGET_YEAR))}.png")
plt.tight_layout()
plt.show()

Line plot analysis for covide period

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- CONFIG ----------
# Primary file: categorical top emotion per topic–platform–year
CSV_TOP = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/top_emotions/top_emotion_per_topic_platform_year.csv"
# Optional alternative (if you have yearly emotion probabilities instead):
CSV_YEARLY = "topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/topic_platform_yearly_emotions.csv"

EMOTIONS_TO_PLOT = ["curiosity", "caring", "admiration"]  # change as needed
COVID_YEARS = (2020, 2021, 2022)
OUT_DIR = "figures_line_timeseries"
ROLLING = 3  # 3-year smoothing; set None to disable
# ----------------------------

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def _standardize_columns(df):
    # Try to standardize common column names
    rename_map = {}
    cols = {c.lower(): c for c in df.columns}

    # Required: platform, year
    for wanted, aliases in {
        "platform": ["platform", "source", "site"],
        "year": ["year", "yr"],
    }.items():
        for a in aliases:
            if a in cols:
                rename_map[cols[a]] = wanted
                break

    # For the categorical file: top_emotion
    for a in ["top_emotion", "emotion", "label", "dominant_emotion"]:
        if a in cols:
            rename_map[cols[a]] = "top_emotion"
            break

    # For the yearly-prob file: emotion + value
    for a in ["top_prob", "probability", "mean_probability", "value", "score"]:
        if a in cols:
            rename_map[cols[a]] = "mean_probability"
            break

    for a in ["emotion", "label", "top_emotion"]:
        if a in cols:
            rename_map[cols[a]] = "emotion"
            break

    df = df.rename(columns=rename_map)
    return df

def plot_from_categorical_top(CSV, emotions, rolling=3):
    df = pd.read_csv(CSV)
    print(df.columns)
    df = _standardize_columns(df)
    print(df.columns)
    # must have: platform, year, top_emotion
    for c in ["platform", "year", "emotion",]:
        if c not in df.columns:
            return False  # not the right file structure

    df = df.dropna(subset=["platform", "year", "emotion"]).copy()
    df["year"] = df["year"].astype(int)

    for emo in emotions:
        d = (
            df.assign(is_target=(df["emotion"].str.lower() == emo.lower()).astype(int))
              .groupby(["platform", "year"])["is_target"].mean()
              .reset_index(name="share")
              .sort_values(["platform", "year"])
        )

        plt.figure(figsize=(9, 5))
        for plat, g in d.groupby("platform", sort=True):
            g = g.sort_values("year")
            x = g["year"].values
            y = g["share"].values
            if rolling and len(g) >= rolling:
                y = pd.Series(y).rolling(rolling, center=True, min_periods=1).mean().values
            plt.plot(x, y, marker="o", label=str(plat))  # no explicit colors

        ymin, ymax = plt.ylim()
        plt.fill_between([COVID_YEARS[0], COVID_YEARS[1], COVID_YEARS[2]], ymin, ymax, alpha=0.12)
        plt.title(f"Share of '{emo}' as Top Emotion over Time")
        plt.xlabel("Year")
        plt.ylabel("Share of groups where emotion is TOP")
        plt.grid(True, alpha=0.3)
        plt.legend(title="Platform", ncols=2, fontsize=9)
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"line_share_top_{emo.replace(' ','_').lower()}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")
    return True

def plot_from_yearly_probs(CSV, emotions, rolling=3):
    df = pd.read_csv(CSV)
    df = _standardize_columns(df)
    # must have: platform, year, emotion, mean_probability
    needed = ["platform", "year", "emotion", "top_prob"]
    if not all(c in df.columns for c in needed):
        return False

    df = df.dropna(subset=needed).copy()
    df["year"] = df["year"].astype(int)
    df["emotion"] = df["emotion"].str.lower()

    for emo in emotions:
        d = df[df["emotion"] == emo.lower()].copy()
        if d.empty:
            print(f"[warn] Emotion '{emo}' not found in {CSV}")
            continue

        plt.figure(figsize=(9, 5))
        for plat, g in d.groupby("platform", sort=True):
            g = g.sort_values("year")
            x = g["year"].values
            y = g["mean_probability"].values
            if rolling and len(g) >= rolling:
                y = pd.Series(y).rolling(rolling, center=True, min_periods=1).mean().values
            plt.plot(x, y, marker="o", label=str(plat))  # no explicit colors

        ymin, ymax = plt.ylim()
        plt.fill_between([COVID_YEARS[0], COVID_YEARS[1]], ymin, ymax, alpha=0.12)
        plt.title(f"Mean Probability of '{emo}' Over Time")
        plt.xlabel("Year")
        plt.ylabel("Mean probability")
        plt.grid(True, alpha=0.3)
        plt.legend(title="Platform", ncols=2, fontsize=9)
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"line_mean_prob_{emo.replace(' ','_').lower()}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")
    return True

# ---------- NEW HELPERS ----------
def _shade_covid(ymin=None, ymax=None):
    import matplotlib.pyplot as plt
    # Shade 2020–2022 inclusive
    if ymin is None or ymax is None:
        ymin, ymax = plt.ylim()
    for year in range(COVID_YEARS[0], COVID_YEARS[-1] + 1):
        plt.axvspan(year - 0.5, year + 0.5, alpha=0.12)

def _smooth(y, window):
    s = pd.Series(y)
    return s.rolling(window, center=True, min_periods=1).mean().values

# ---------- FIX: yearly-probs column check ----------
def plot_from_yearly_probs(CSV, emotions, rolling=3):
    df = pd.read_csv(CSV)
    df = _standardize_columns(df)
    # must have: platform, year, emotion, mean_probability
    needed = ["platform", "year", "emotion", "mean_probability"]  # <-- fixed
    if not all(c in df.columns for c in needed):
        return False

    df = df.dropna(subset=needed).copy()
    df["year"] = df["year"].astype(int)
    df["emotion"] = df["emotion"].str.lower()

    for emo in emotions:
        d = df[df["emotion"] == emo.lower()].copy()
        if d.empty:
            print(f"[warn] Emotion '{emo}' not found in {CSV}")
            continue

        plt.figure(figsize=(9, 5))
        for plat, g in d.groupby("platform", sort=True):
            g = g.sort_values("year")
            x = g["year"].values
            y = g["mean_probability"].values
            if rolling and len(g) >= rolling:
                y = _smooth(y, rolling)
            plt.plot(x, y, marker="o", label=str(plat))

        _shade_covid()
        plt.title(f"Mean Probability of '{emo}' Over Time")
        plt.xlabel("Year")
        plt.ylabel("Mean probability")
        plt.grid(True, alpha=0.3)
        plt.legend(title="Platform", ncols=2, fontsize=9)
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"line_mean_prob_{emo.replace(' ','_').lower()}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")
    return True

# ---------- NEW: multi-emotion plots (categorical: share as TOP) ----------
def plot_multi_emotions_per_platform_from_categorical(
    CSV, top_k=12, rolling=3, min_years=3
):
    df = pd.read_csv(CSV)
    df = _standardize_columns(df)
    needed = ["platform", "year", "emotion"]
    if not all(c in df.columns for c in needed):
        return False

    df = df.dropna(subset=needed).copy()
    df["year"] = df["year"].astype(int)
    df["emotion"] = df["emotion"].str.lower()

    # compute share that emotion was TOP per platform-year
    share = (
        df.assign(one=1)
          .groupby(["platform", "year", "emotion"])["one"].mean()
          .reset_index(name="share")
    )

    for plat, dplat in share.groupby("platform", sort=True):
        # pick top-K emotions by total share over all years on this platform
        top_emotions = (
            dplat.groupby("emotion")["share"].sum()
                 .sort_values(ascending=False)
                 .head(top_k)
                 .index.tolist()
        )
        dkeep = dplat[dplat["emotion"].isin(top_emotions)].copy()

        if dkeep["year"].nunique() < min_years:
            continue

        plt.figure(figsize=(11, 6))
        for emo, g in dkeep.groupby("emotion", sort=False):
            g = g.sort_values("year")
            x = g["year"].values
            y = g["share"].values
            if rolling and len(g) >= rolling:
                y = _smooth(y, rolling)
            # many emotions -> thin lines, markers help track
            plt.plot(x, y, marker="o", linewidth=1.2, label=emo)

        _shade_covid()
        plt.title(f"{plat}: Share of Emotions Being TOP (Top {len(top_emotions)})")
        plt.xlabel("Year")
        plt.ylabel("Share of groups where emotion is TOP")
        plt.grid(True, alpha=0.3)
        # large legend: move outside
        plt.legend(title="Emotion", fontsize=8, ncols=2, loc="upper left", bbox_to_anchor=(1.02, 1))
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"multi_line_top_share_{plat}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")
    return True

# ---------- NEW: multi-emotion plots (yearly mean probabilities) ----------
def plot_multi_emotions_per_platform_from_yearly_probs(
    CSV, top_k=12, rolling=3, min_years=3
):
    df = pd.read_csv(CSV)
    df = _standardize_columns(df)
    needed = ["platform", "year", "emotion", "mean_probability"]
    if not all(c in df.columns for c in needed):
        return False

    df = df.dropna(subset=needed).copy()
    df["year"] = df["year"].astype(int)
    df["emotion"] = df["emotion"].str.lower()

    # average probability per platform-year-emotion
    grp = (df.groupby(["platform", "year", "emotion"])["mean_probability"]
             .mean()
             .reset_index(name="prob"))

    for plat, dplat in grp.groupby("platform", sort=True):
        # pick top-K emotions by average prob across the whole period
        top_emotions = (
            dplat.groupby("emotion")["prob"].mean()
                 .sort_values(ascending=False)
                 .head(top_k)
                 .index.tolist()
        )
        dkeep = dplat[dplat["emotion"].isin(top_emotions)].copy()

        if dkeep["year"].nunique() < min_years:
            continue

        plt.figure(figsize=(11, 6))
        for emo, g in dkeep.groupby("emotion", sort=False):
            g = g.sort_values("year")
            x = g["year"].values
            y = g["prob"].values
            if rolling and len(g) >= rolling:
                y = _smooth(y, rolling)
            plt.plot(x, y, marker="o", linewidth=1.2, label=emo)

        _shade_covid()
        plt.title(f"{plat}: Mean Emotion Probability (Top {len(top_emotions)})")
        plt.xlabel("Year")
        plt.ylabel("Mean probability")
        plt.grid(True, alpha=0.3)
        plt.legend(title="Emotion", fontsize=8, ncols=2, loc="upper left", bbox_to_anchor=(1.02, 1))
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"multi_line_mean_prob_{plat}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")
    return True

def plot_multi_emotions_per_platform_from_categorical_share(
    CSV, top_k=12, rolling=3, min_years=3
):
    df = pd.read_csv(CSV)
    df = _standardize_columns(df)
    needed = ["platform", "year", "emotion"]  # emotion == top_emotion in this file
    if not all(c in df.columns for c in needed):
        return False

    df = df.dropna(subset=needed).copy()
    df["year"] = df["year"].astype(int)
    df["emotion"] = df["emotion"].str.lower()

    # Count rows per platform-year (denominator) and per platform-year-emotion (numerator)
    denom = (df.groupby(["platform", "year"])
               .size()
               .reset_index(name="n_total"))

    numer = (df.groupby(["platform", "year", "emotion"])
               .size()
               .reset_index(name="n_emotion"))

    share = numer.merge(denom, on=["platform", "year"], how="left")
    share["share"] = share["n_emotion"] / share["n_total"]

    for plat, dplat in share.groupby("platform", sort=True):
        # Pick top-K emotions by total presence across years on THIS platform
        top_emotions = (dplat.groupby("emotion")["n_emotion"]
                            .sum()
                            .sort_values(ascending=False)
                            .head(top_k)
                            .index.tolist())
        dkeep = dplat[dplat["emotion"].isin(top_emotions)].copy()
        if dkeep["year"].nunique() < min_years:
            continue

        plt.figure(figsize=(11, 6))
        for emo, g in dkeep.groupby("emotion", sort=False):
            g = g.sort_values("year")
            x = g["year"].values
            y = g["share"].values
            if rolling and len(g) >= rolling:
                y = _smooth(y, rolling)
            plt.plot(x, y, marker="o", linewidth=1.2, label=emo)

        _shade_covid()
        plt.title(f"{plat}: Share of Emotions Being TOP (Top {len(top_emotions)})")
        plt.xlabel("Year")
        plt.ylabel("Share of groups where emotion is TOP")
        plt.grid(True, alpha=0.3)
        plt.legend(title="Emotion", fontsize=8, ncols=2, loc="upper left", bbox_to_anchor=(1.02, 1))
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"multi_line_top_share_{plat}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")
    return True

# ---------- CALL ONE OF THESE ----------
# 1) If you want multi-emotion charts from the categorical “top emotion” file:
# plot_multi_emotions_per_platform_from_categorical(CSV_TOP, top_k=14, rolling=ROLLING)

# 2) Or, from the yearly probability file (if you have it):
# plot_multi_emotions_per_platform_from_yearly_probs(CSV_YEARLY, top_k=4, rolling=ROLLING)

plot_multi_emotions_per_platform_from_categorical_share(
    CSV_TOP, top_k=14, rolling=ROLLING
)

# Try categorical first; if not suitable, try yearly-probs
ok = plot_from_categorical_top(CSV_TOP, EMOTIONS_TO_PLOT, rolling=ROLLING)
if not ok and os.path.exists(CSV_YEARLY):
    print("[info] Falling back to yearly probabilities file...")
    ok = plot_from_yearly_probs(CSV_YEARLY, EMOTIONS_TO_PLOT, rolling=ROLLING)

if not ok:
    print("❌ Could not plot. Please check that the CSV has the expected columns.")

Line plots for the covid analysis (per-doc)

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import to_hex

# =================== CONFIG ===================
PER_DOC_CSV = "topic_sentiment_and_emotions_with_per_doc_emotions/topic_sentiment_highest_probability_emotions.csv"
OUT_DIR  = "topic_sentiment_and_emotions_with_per_doc_emotions/figures/covid"

EMOTIONS_TO_PLOT = ["curiosity", "caring", "admiration"]   # change as needed
ROLLING = 3            # 3-year centered smoothing; set to None to disable
COVID_YEARS = (2020, 2021, 2022)

# Plot sizing + y-axis zoom (use None to auto)
FIGSIZE = (12, 9)      # taller by default
Y_ZOOM_SHARE = None    # e.g., (0.05, 0.45) to zoom for shares
Y_ZOOM_PROB  = None    # e.g., (0.62, 0.88) to zoom for probabilities

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# =================== HELPERS ===================
def _standardize_perdoc_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Map common column aliases to canonical names: platform, year, emotion (top), top_prob."""
    cols = {c.lower(): c for c in df.columns}
    rename = {}

    # platform / year
    for want, aliases in {
        "platform": ["platform", "source", "site"],
        "year":     ["year", "yr"],
    }.items():
        for a in aliases:
            if a in cols:
                rename[cols[a]] = want
                break

    # top emotion label (categorical)
    for a in ["top_emotion", "emotion", "label", "dominant_emotion"]:
        if a in cols:
            rename[cols[a]] = "top_emotion"
            break

    # top probability
    for a in ["top_prob", "top_emotion_prob", "probability", "max_prob", "pred_prob", "score"]:
        if a in cols:
            rename[cols[a]] = "top_prob"
            break

    df = df.rename(columns=rename)
    return df

def _smooth(y, window):
    s = pd.Series(y)
    return s.rolling(window, center=True, min_periods=1).mean().values

def _shade_covid():
    # Shade 2020–2022 inclusive
    for year in range(COVID_YEARS[0], COVID_YEARS[-1] + 1):
        plt.axvspan(year - 0.5, year + 0.5, alpha=0.12)

def _make_distinct_palette(n, seed=7):
    """A long, distinct palette; deterministic order."""
    base_names = ["tab20", "tab20b", "tab20c", "Dark2", "Set1", "Set2", "Set3", "Paired", "Accent", "Pastel1"]
    colors = []
    for name in base_names:
        cmap = mpl.cm.get_cmap(name)
        if hasattr(cmap, "colors"):
            colors.extend([to_hex(c) for c in cmap.colors])
        else:
            colors.extend([to_hex(cmap(i / (cmap.N - 1))) for i in range(cmap.N)])
    # de-dupe
    seen = set(); uniq = []
    for c in colors:
        if c not in seen:
            seen.add(c); uniq.append(c)
    # separate similar hues deterministically
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(uniq))
    uniq = [uniq[i] for i in order]
    if n <= len(uniq):
        return uniq[:n]
    return (uniq * ((n // len(uniq)) + 1))[:n]

# =================== LOAD PER-DOC ===================
df = pd.read_csv(PER_DOC_CSV)
df = _standardize_perdoc_columns(df)

needed = ["platform", "year", "top_emotion"]
if not all(c in df.columns for c in needed):
    raise ValueError(f"Per-doc CSV must include columns {needed}. Found: {list(df.columns)}")

# Clean
df = df.dropna(subset=["platform", "year", "top_emotion"]).copy()
df["platform"] = df["platform"].astype(str)
df["top_emotion"] = df["top_emotion"].astype(str).str.lower()
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["year"]).copy()
df["year"] = df["year"].astype(int)

# Optional: weight per row (if you have a 'count' column)
if "count" in df.columns:
    df["_w"] = pd.to_numeric(df["count"], errors="coerce").fillna(0.0)
else:
    df["_w"] = 1.0

# Fixed, distinct colors for all emotions (global mapping)
ALL_EMOTIONS = sorted(df["top_emotion"].unique().tolist())
_color_list = _make_distinct_palette(len(ALL_EMOTIONS))
EMO_COLOR = {emo: _color_list[i] for i, emo in enumerate(ALL_EMOTIONS)}

# =================== 1) SINGLE-EMOTION LINES: SHARE OF DOCS WHERE TOP == EMO ===================
def plot_from_perdoc_top_share(emotions, rolling=3, y_zoom=None):
    for emo in emotions:
        emo_l = emo.lower()
        # numerator: weighted docs where top_emotion == emo
        numer = (
            df.assign(is_target=(df["top_emotion"] == emo_l).astype(int))
              .groupby(["platform", "year"], dropna=False)
              .apply(lambda g: np.sum(g["is_target"] * g["_w"]))
              .reset_index(name="n_target")
        )
        # denominator: all docs (weighted) per platform-year
        denom = (
            df.groupby(["platform", "year"], dropna=False)["_w"]
              .sum()
              .reset_index(name="n_total")
        )
        d = numer.merge(denom, on=["platform", "year"], how="left")
        d["share"] = d["n_target"] / d["n_total"].replace(0, np.nan)

        plt.figure(figsize=FIGSIZE)
        for plat, g in d.groupby("platform", sort=True):
            g = g.sort_values("year")
            x = g["year"].values
            y = g["share"].values
            if rolling and len(g) >= rolling:
                y = _smooth(y, rolling)
            plt.plot(x, y, marker="o", linewidth=2, label=str(plat))

        _shade_covid()
        plt.title(f"Share of documents with TOP emotion = '{emo_l}'")
        plt.xlabel("Year")
        plt.ylabel("Share of documents")
                # Y-axis: explicit zoom or auto-zoom with margin
        if y_zoom is not None:
            plt.ylim(*y_zoom)
        else:
            yvals = d["share"].to_numpy(dtype=float)
            yvals = yvals[np.isfinite(yvals)]
            if yvals.size:
                y_min, y_max = float(np.nanmin(yvals)), float(np.nanmax(yvals))
                pad = max(0.02, (y_max - y_min) * 0.25)
                plt.ylim(max(0.0, y_min - pad), min(1.0, y_max + pad))
            else:
                plt.ylim(0, 1)
        plt.grid(alpha=0.3)
        plt.legend(title="Platform", ncols=2, fontsize=9, bbox_to_anchor=(1.02,1), loc="upper left")
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"line_share_top_{emo_l.replace(' ','_')}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")

# =================== 2) MULTI-EMOTION (PER PLATFORM): SHARE OF DOCS WHERE TOP == EMO ===================
def plot_multi_emotions_per_platform_from_perdoc_top_share(top_k=12, rolling=3, min_years=3, y_zoom=None):
    # Compute share per platform-year-emotion
    numer = (
        df.groupby(["platform", "year", "top_emotion"], dropna=False)["_w"]
          .sum().reset_index(name="n_emotion")
    )
    denom = (
        df.groupby(["platform", "year"], dropna=False)["_w"]
          .sum().reset_index(name="n_total")
    )
    share = numer.merge(denom, on=["platform", "year"], how="left")
    share["share"] = share["n_emotion"] / share["n_total"].replace(0, np.nan)

    for plat, dplat in share.groupby("platform", sort=True):
        # pick top-K emotions by total weighted presence on this platform
        top_emotions = (
            dplat.groupby("top_emotion")["n_emotion"].sum()
                 .sort_values(ascending=False).head(top_k).index.tolist()
        )
        dkeep = dplat[dplat["top_emotion"].isin(top_emotions)].copy()
        if dkeep["year"].nunique() < min_years:
            continue

        # consistent color order
        col_order = top_emotions
        plt.figure(figsize=FIGSIZE)
        for emo in col_order:
            g = dkeep[dkeep["top_emotion"] == emo].sort_values("year")
            if g.empty:
                continue
            x = g["year"].values
            y = g["share"].values
            if rolling and len(g) >= rolling:
                y = _smooth(y, rolling)
            plt.plot(x, y, marker="o", linewidth=1.6, label=emo, color=EMO_COLOR.get(emo, None))

        _shade_covid()
        plt.title(f"{plat}: Share of Top Emotions (Top {len(col_order)})")
        plt.xlabel("Year")
        plt.ylabel("Share of documents")
        if y_zoom is not None:
            plt.ylim(*y_zoom)
        else:
            yvals = dkeep["share"].to_numpy(dtype=float)
            yvals = yvals[np.isfinite(yvals)]
            if yvals.size:
                y_min, y_max = float(np.nanmin(yvals)), float(np.nanmax(yvals))
                pad = max(0.02, (y_max - y_min) * 0.25)
                plt.ylim(max(0.0, y_min - pad), min(1.0, y_max + pad))
            else:
                plt.ylim(0, 1)
        plt.grid(True, alpha=0.3)
        plt.legend(title="Emotion", fontsize=8, ncols=2, loc="upper left", bbox_to_anchor=(1.02, 1))
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"multi_line_top_share_{plat}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")

# =================== 3) MULTI-EMOTION (PER PLATFORM): MEAN TOP PROBABILITY ===================
def plot_multi_emotions_per_platform_from_perdoc_mean_prob(top_k=12, rolling=3, min_years=3, y_zoom=None):
    if "top_prob" not in df.columns:
        raise ValueError("Per-doc CSV must include 'top_prob' to plot mean probabilities.")
    dprob = df.dropna(subset=["top_prob"]).copy()
    dprob["top_prob"] = pd.to_numeric(dprob["top_prob"], errors="coerce")

    grp = (
        dprob.groupby(["platform", "year", "top_emotion"], dropna=False)["top_prob"]
             .mean().reset_index(name="mean_prob")
    )

    for plat, dplat in grp.groupby("platform", sort=True):
        # choose top-K by average mean_prob across period (or switch to volume with .size())
        top_emotions = (
            dplat.groupby("top_emotion")["mean_prob"].mean()
                 .sort_values(ascending=False).head(top_k).index.tolist()
        )
        dkeep = dplat[dplat["top_emotion"].isin(top_emotions)].copy()
        if dkeep["year"].nunique() < min_years:
            continue

        plt.figure(figsize=FIGSIZE)
        for emo in top_emotions:
            g = dkeep[dkeep["top_emotion"] == emo].sort_values("year")
            x = g["year"].values
            y = g["mean_prob"].values
            if rolling and len(g) >= rolling:
                y = _smooth(y, rolling)
            plt.plot(x, y, marker="o", linewidth=1.6, label=emo, color=EMO_COLOR.get(emo, None))

        _shade_covid()
        plt.title(f"{plat}: Mean Top-Emotion Probability (Top {len(top_emotions)})")
        plt.xlabel("Year")
        plt.ylabel("Mean probability of top emotion")
        if y_zoom is not None:
            plt.ylim(*y_zoom)
        else:
            # auto-zoom around data range for readability
            y_min, y_max = dkeep["mean_prob"].min(), dkeep["mean_prob"].max()
            if pd.notnull(y_min) and pd.notnull(y_max):
                margin = max(0.02, (y_max - y_min) * 0.25)
                plt.ylim(max(0, y_min - margin), min(1, y_max + margin))
        plt.grid(True, alpha=0.3)
        plt.legend(title="Emotion", fontsize=8, ncols=2, loc="upper left", bbox_to_anchor=(1.02, 1))
        plt.tight_layout()
        out = os.path.join(OUT_DIR, f"multi_line_mean_prob_{plat}.png")
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out}")

# =================== CALLS ===================
# (A) Single-emotion, per-platform lines: share of docs where top == emotion
plot_from_perdoc_top_share(EMOTIONS_TO_PLOT, rolling=ROLLING, y_zoom=Y_ZOOM_SHARE)

# (B) Multi-emotion per platform: shares
plot_multi_emotions_per_platform_from_perdoc_top_share(
    top_k=5, rolling=ROLLING, min_years=3, y_zoom=Y_ZOOM_SHARE
)

# (C) Multi-emotion per platform: mean top_prob
plot_multi_emotions_per_platform_from_perdoc_mean_prob(
    top_k=5, rolling=ROLLING, min_years=3, y_zoom=Y_ZOOM_PROB
)

Pine plot of covid analysis. See if it really was the cause.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- CONFIG ----------
CSV_DOC   = "topic_sentiment_and_emotions_with_per_doc_emotions/topic_sentiment_highest_probability_emotions.csv"
OUT_DIR   = "topic_sentiment_and_emotions_with_per_doc_emotions/figures/ridgeline"
PLATFORMS = ["Reddit", "YouTube", "News", "PubMed"]     # choose any subset
YEAR_RANGE = None                                        # e.g., (2014, 2025) or None for all
COVID_YEARS = (2020, 2021, 2022)                               # highlighted on the plot
BINS = 40
Y_GAP = 1.05                                             # vertical spacing between ridges
SHOW_MEAN = True                                         # draw a vertical mean marker per year
# ----------------------------

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def _standardize_columns(df):
    rename_map = {}
    cols = {c.lower(): c for c in df.columns}
    mapping = {
        "platform": ["platform", "source", "site", "_platform"],
        "year":     ["year", "yr", "_year"],
        "top_emotion_prob": ["top_emotion_prob", "max_emotion_prob", "top_prob", "probability", "score"]
    }
    for want, aliases in mapping.items():
        for a in aliases:
            if a in cols:
                rename_map[cols[a]] = want
                break
    return df.rename(columns=rename_map)

def ridgeline_by_year(
    df, platform=None, year_range=None, bins=40, y_gap=1.05,
    title=None, outfile=None, covid_years=(2020, 2021, 2022), show_mean=True
):
    d = df.copy()
    if platform is not None:
        d = d[d["platform"] == platform].copy()
    if year_range:
        d = d[(d["year"] >= year_range[0]) & (d["year"] <= year_range[1])].copy()

    # years sorted ascending
    years = sorted(d["year"].dropna().astype(int).unique())
    if not years:
        print(f"[warn] No years available for platform={platform}.")
        return

    # Common x-grid in [0, 1] (probabilities)
    x = np.linspace(0, 1, 400)

    # Compute densities + means per year
    max_height = 0.0
    curves, means = {}, {}
    for y in years:
        vals = d.loc[d["year"] == y, "top_emotion_prob"].dropna().values
        if vals.size == 0:
            dens = np.zeros_like(x)
            mu = np.nan
        else:
            counts, edges = np.histogram(vals, bins=bins, range=(0, 1), density=True)
            centers = 0.5 * (edges[:-1] + edges[1:])
            dens = np.interp(x, centers, counts, left=0, right=0)
            mu = float(np.mean(vals))
        curves[y] = dens
        means[y] = mu
        if dens.max() > max_height:
            max_height = dens.max() if dens.max() > 0 else max_height

    # Figure height scales with number of years
    fig = plt.figure(figsize=(10, max(4.5, 0.35 * len(years))))
    ax = plt.gca()

    # Plot COVID band(s) behind ridges (by y-span for those years)
    if covid_years:
        for y in years:
            if covid_years[0] <= y <= covid_years[2]:
                i = years.index(y)
                base = i * y_gap
                ax.axhspan(base, base + 1.0, alpha=0.12)  # soft highlight, no color set

    # Draw ridges
    for i, y in enumerate(years):
        dens = curves[y]
        dens_norm = dens / (max_height if max_height > 0 else 1.0)
        baseline = i * y_gap
        ax.fill_between(x, baseline, baseline + dens_norm, alpha=0.6, linewidth=0)   # fill
        ax.plot(x, baseline + dens_norm, linewidth=1)                                # outline
        # Mean marker
        if show_mean and np.isfinite(means[y]):
            ax.plot([means[y], means[y]], [baseline, baseline + 1.02], linestyle="--", linewidth=1)

    # Axes + labels
    ax.set_xlim(0, 1)
    ax.set_xlabel("Top-emotion probability (model confidence)")
    ax.set_yticks([i * y_gap + 0.5 for i in range(len(years))])
    ax.set_yticklabels([str(y) for y in years])
    ttl = title if title else "Ridgeline ('Pine') of Top-Emotion Probability by Year"
    if platform:
        ttl += f" — {platform}"
    ax.set_title(ttl)
    ax.grid(axis="x", alpha=0.25)

    # On-figure explanation panel (how to read the plot)
    #expl = (
    #    "What this shows:\n"
    #    "• Each ridge = distribution of per-document top-emotion probabilities in a year.\n"
    #    "• Right-shifted ridges ⇒ higher model confidence / clearer emotional signal.\n"
    #    "• Tall, narrow ridges ⇒ many docs near the same confidence (homogeneous tone).\n"
    #    "• Flat/broad ridges ⇒ mixed or ambiguous emotions (heterogeneous tone).\n"
    #    f"• Shaded bands mark COVID years ({covid_years[0]}–{covid_years[2]}). "
    #    "Dashed line = yearly mean."
    #)
    # Place as fig-level text so it doesn't overlap; wrap width by setting a bbox
    #fig.text(
    #    0.02, 0.02, expl,
    #    ha="left", va="bottom", fontsize=9,
    #    bbox=dict(boxstyle="round,pad=0.35", alpha=0.08, lw=0)
    #)

    plt.tight_layout(rect=(0, 0.08, 1, 1))
    if outfile:
        plt.savefig(outfile, dpi=300, bbox_inches="tight")
        print(f"Saved: {outfile}")
    plt.show()

# ---- LOAD & STANDARDIZE ----
df_docs = pd.read_csv(CSV_DOC)
df_docs = _standardize_columns(df_docs)

required = {"platform", "year", "top_emotion_prob"}
if not required.issubset(df_docs.columns):
    raise ValueError(f"CSV must contain columns {required}, got {df_docs.columns.tolist()}")

df_docs["year"] = df_docs["year"].astype(int)

# ---- RUN FOR CHOSEN PLATFORMS ----
for plat in PLATFORMS:
    outpath = os.path.join(OUT_DIR, f"ridgeline_{plat.replace(' ','_')}.png")
    ridgeline_by_year(
        df_docs,
        platform=plat,
        year_range=YEAR_RANGE,
        bins=BINS,
        y_gap=Y_GAP,
        title="Ridgeline of Top-Emotion Probability by Year",
        outfile=outpath,
        covid_years=COVID_YEARS,
        show_mean=SHOW_MEAN,
    )

In [ ]:
import pandas as pd

df = pd.read_csv("topic_sentiment_and_emotions_with_new_pubmed_youtube_data_use_doc_average/topic_platform_yearly_emotions.csv")
print(df[["platform","year"]].drop_duplicates().sort_values(["platform","year"]).head(30))
print(df.groupby("platform")["year"].nunique())

---
<!-- nav-footer -->

[Project Overview](../../PROJECT_OVERVIEW.ipynb)
